# Real GPT-2 heads: one-shot vs. multi-round refinement

**Research question (Next-steps #1 from the [README](../README.md)):** does
**feedback-driven multi-round refinement** produce better and *less-degenerate*
attention-program explanations than the paper's **one-shot** synthesis — on
*real* transformer heads?

This is the experiment that turns the working system into a *result*: it
extracts real attention heads from **GPT-2**, has a code LM (**Qwen3-4B**) write
`predict_attention` programs for each, and compares:

- **one-shot** best-of-N (the paper's baseline), vs.
- **N-round refinement** where each round is shown the env's structured
  worst-edge feedback (`build_refinement_prompt`) and tries again,

scored by the verifiable IoU / JSD / degeneracy metrics — no LM judge. It
reports a metrics table across many heads plus a per-round improvement curve.

**Inference-only — no RL training** — so it's cheap and robust; it needs a GPU
only to run GPT-2 (extraction) and Qwen3-4B (synthesis). Self-contained: it
writes the whole `attn_program_rl` package to disk, same as the main demo.

## 1. Install & write the package

Same setup as the main notebook: text-only stack, no torch upgrade, torchvision
removed (avoids the ABI-mismatch import crash), then the package written to
disk.

In [ ]:
%pip install -q -U "transformers>=4.51" peft accelerate datasets numpy scipy matplotlib pandas pytest
%pip uninstall -y torchvision

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0),
          round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
import os
for d in ["env", "grpo", "scripts", "tests"]:
    os.makedirs(d, exist_ok=True)
print("package dirs ready")

In [ ]:
%%writefile env/__init__.py
from .attention_env import Action, AttentionProgramEnv, Observation, StepResult
from .data import AttentionExample, load_dataset, split_train_val, synthetic_head_dataset, group_by_head
from .executor import compile_program, run_program, run_program_inproc
from .reward import compute_reward, iou_score, jsd_score, analyze_complexity, positional_collapse_score

__all__ = [
    "Action", "AttentionProgramEnv", "Observation", "StepResult",
    "AttentionExample", "load_dataset", "split_train_val", "synthetic_head_dataset", "group_by_head",
    "compile_program", "run_program", "run_program_inproc",
    "compute_reward", "iou_score", "jsd_score", "analyze_complexity", "positional_collapse_score",
]


In [ ]:
%%writefile env/reward.py
"""
Reward functions for RL-based attention-program synthesis.

Implements the two metrics from Hayes, Li & Andreas (2026), "Explaining
Attention with Program Synthesis" (arXiv:2606.19317):

  - IoU(A, Ahat)  = sum(min(A,Ahat)) / sum(max(A,Ahat))          [paper eq. 3]
  - JSD(A, Ahat)  = 0.5*KL(A||M) + 0.5*KL(Ahat||M), M=(A+Ahat)/2  [paper eq. 2]

plus a complexity penalty that the paper's pipeline does *not* have, and
which is the main thing this environment adds. The paper's own limitations
section names two failure modes we're explicitly guarding against:

  1. "many high-scoring programs are not particularly complex ... the
     improvement in QA seen ... at low replacement levels may be akin to
     a pruning effect" -> a policy optimizing IoU alone can collapse onto
     degenerate solutions (e.g. "always attend to token 0") that fit the
     coarse metric without capturing real head logic.
  2. Non-well-formed / non-executable programs are assigned maximal
     divergence (the paper's rule) -> we mirror that as a hard reward floor.

Reward = w_iou * IoU - w_jsd * JSD - w_cx * complexity_penalty(program)
with a hard floor for syntax/execution failures.
"""
from __future__ import annotations

import ast
import dataclasses
import numpy as np


EPS = 1e-10


# --------------------------------------------------------------------------
# Core similarity metrics (paper-faithful)
# --------------------------------------------------------------------------

def _as_nonneg_matrix(M: np.ndarray) -> np.ndarray:
    M = np.asarray(M, dtype=np.float64)
    M = np.clip(M, 0.0, None)
    return M


def iou_score(A: np.ndarray, Ahat: np.ndarray) -> float:
    """Intersection-over-Union between two nonnegative attention matrices.

    Paper eq. 3: IoU(A, Ahat) = sum_ij min(A_ij, Ahat_ij) / sum_ij max(A_ij, Ahat_ij)
    """
    A = _as_nonneg_matrix(A)
    Ahat = _as_nonneg_matrix(Ahat)
    if A.shape != Ahat.shape:
        return 0.0
    inter = np.minimum(A, Ahat).sum()
    union = np.maximum(A, Ahat).sum()
    if union < EPS:
        return 1.0 if inter < EPS else 0.0
    return float(inter / union)


def _normalize_to_distribution(M: np.ndarray) -> np.ndarray:
    M = _as_nonneg_matrix(M)
    total = M.sum()
    if total < EPS:
        # degenerate -> uniform distribution, treated as maximally uninformative
        return np.full_like(M, 1.0 / M.size)
    return M / total


def jsd_score(A: np.ndarray, Ahat: np.ndarray) -> float:
    """Jensen-Shannon distance between two attention matrices treated as
    flattened joint distributions over token pairs.

    Paper eq. 2: JSD(A, Ahat) = 0.5*KL(A||M) + 0.5*KL(Ahat||M), M = (A+Ahat)/2
    Returns the JS *divergence* (bounded in [0, ln 2]); callers that want the
    JS *distance* (bounded in [0, 1]) should take sqrt().
    """
    if A.shape != Ahat.shape:
        return float(np.log(2.0))  # maximal divergence for shape mismatch
    P = _normalize_to_distribution(A)
    Q = _normalize_to_distribution(Ahat)
    M = 0.5 * (P + Q)
    kl_pm = np.sum(P * (np.log(P + EPS) - np.log(M + EPS)))
    kl_qm = np.sum(Q * (np.log(Q + EPS) - np.log(M + EPS)))
    jsd = 0.5 * kl_pm + 0.5 * kl_qm
    return float(max(jsd, 0.0))


# --------------------------------------------------------------------------
# Complexity penalty
# --------------------------------------------------------------------------

# AST node types that count as "real" logic vs. boilerplate. Kept small and
# legible on purpose -- this is a knob you should tune per experiment.
_LOGIC_NODES = (
    ast.If, ast.For, ast.While, ast.BoolOp, ast.Compare,
    ast.FunctionDef, ast.Call, ast.ListComp, ast.DictComp,
    ast.IfExp, ast.Lambda,
)


@dataclasses.dataclass
class ComplexityReport:
    node_count: int
    logic_node_count: int
    is_trivial_constant: bool
    collapse_score: float  # 0 = attention spread across columns, 1 = collapsed onto one fixed column
    penalty: float


def positional_collapse_score(Ahat: np.ndarray) -> float:
    """Intrinsic degeneracy signal computed directly from a candidate's own
    output -- no ground truth, no second execution needed.

    A program that ignores its input and always routes attention to one
    fixed column (e.g. `attention[:, 0] = 1.0`) produces a column-marginal
    distribution (mean attention received by each column, across all query
    rows) that's a spike. A program with real positional or content logic
    -- even a *purely* positional one like "attend to token i-1", which is
    legitimate and shouldn't be penalized -- spreads attention across many
    columns as the query index moves, so its column marginal is closer to
    uniform. We measure this via normalized Shannon entropy of the column
    marginal: 1 - H(marginal)/log(n).

    Verified against the four synthetic archetypes in env/data.py:
    previous-token and diagonal-self (both purely positional, both
    legitimate) score ~0.04 and ~0.00 collapse (i.e. NOT flagged); a
    constant first-token program scores ~0.998 collapse (correctly
    flagged). This is deliberately behavioral rather than AST-based: no
    static pattern-match on the source catches `attention[:, 0] = 1.0`,
    since it has no conditionals to match on at all.
    """
    n = Ahat.shape[0]
    if n <= 1:
        return 0.0
    col_marginal = np.clip(Ahat, 0, None).mean(axis=0)
    total = col_marginal.sum()
    if total < EPS:
        return 1.0  # all-zero output is maximally degenerate
    col_marginal = col_marginal / total
    entropy = -np.sum(col_marginal * np.log(col_marginal + EPS))
    max_entropy = np.log(n)
    normalized_entropy = entropy / max_entropy if max_entropy > EPS else 1.0
    return float(np.clip(1.0 - normalized_entropy, 0.0, 1.0))


def analyze_complexity(code: str, min_nodes: int = 6, target_nodes: int = 40,
                        max_nodes: int = 220,
                        collapse_score: float = 0.0) -> ComplexityReport:
    """Score a candidate program's complexity and structural triviality.

    Penalizes two failure modes:
      - too trivial (below `min_nodes`) or too complex (above `max_nodes`,
        shallow U around `target_nodes`) -- discourages both no-op programs
        and unreadable ones that overfit rather than describe general logic.
      - positional collapse (see `positional_collapse_score`): this is
        precisely the "pruning effect" confound the paper's own limitations
        section flags -- high-IoU programs that fit the metric by ignoring
        the query entirely rather than capturing real head logic. Pass the
        candidate's own output through `positional_collapse_score` and feed
        it in here; this function alone can't compute it from source.
    """
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return ComplexityReport(0, 0, True, 1.0, penalty=1.0)

    node_count = sum(1 for _ in ast.walk(tree))
    logic_count = sum(1 for n in ast.walk(tree) if isinstance(n, _LOGIC_NODES))
    is_trivial_constant = node_count < min_nodes

    if node_count <= min_nodes:
        penalty = 1.0
    elif node_count >= max_nodes:
        overshoot = (node_count - max_nodes) / max(target_nodes, 1)
        penalty = min(1.0, 0.3 + 0.1 * overshoot)
    else:
        # shallow U, 0 at target, rising toward both edges, capped well below 1
        span = max(target_nodes - min_nodes, max_nodes - target_nodes, 1)
        dist = abs(node_count - target_nodes) / span
        penalty = min(0.35, 0.35 * dist)

    # collapse_score in [0,1] scales an additional penalty up to 0.6 at full
    # collapse -- large enough to flip the ranking against a comparably-fit
    # program that actually varies its attention with the query.
    penalty = max(penalty, 0.6 * collapse_score)

    return ComplexityReport(node_count, logic_count, is_trivial_constant,
                             collapse_score, penalty=penalty)


# --------------------------------------------------------------------------
# Combined reward
# --------------------------------------------------------------------------

@dataclasses.dataclass
class RewardBreakdown:
    reward: float
    iou: float
    jsd: float
    complexity_penalty: float
    executable: bool
    error: str | None = None


DEFAULT_WEIGHTS = dict(w_iou=1.0, w_jsd=0.5, w_cx=0.3)
FAILURE_REWARD = -1.0  # paper's "non-well-formed -> maximal divergence" rule


def compute_reward(A: np.ndarray | None, Ahat: np.ndarray | None, code: str,
                    executable: bool, error: str | None = None,
                    weights: dict | None = None,
                    collapse_score: float = 0.0) -> RewardBreakdown:
    w = {**DEFAULT_WEIGHTS, **(weights or {})}

    if not executable or Ahat is None:
        return RewardBreakdown(FAILURE_REWARD, 0.0, float(np.log(2.0)), 1.0,
                                executable=False, error=error)

    iou = iou_score(A, Ahat)
    jsd = jsd_score(A, Ahat)
    cx = analyze_complexity(code, collapse_score=collapse_score).penalty

    reward = w["w_iou"] * iou - w["w_jsd"] * jsd - w["w_cx"] * cx
    return RewardBreakdown(reward=float(reward), iou=iou, jsd=jsd,
                            complexity_penalty=cx, executable=True, error=None)


In [ ]:
%%writefile env/executor.py
"""
Sandboxed execution of policy-generated attention programs.

Contract each candidate program must satisfy (mirrors the paper's program
signature, X -> A):

    def predict_attention(tokens: list[str]) -> np.ndarray:
        n = len(tokens)
        attention = np.zeros((n, n))
        ...
        return attention

Programs get numpy (as `np`) and `re` in scope, matching the paper's stated
tool access (NumPy / spaCy / NLTK) minus the heavier NLP libraries -- add
those back in `SAFE_GLOBALS` if your synthesis prompt advertises them, but
note each addition widens the sandbox surface and should be reviewed.

Safety notes (this is a research sandbox, not a security boundary):
  - AST-allowlists imports/calls before exec so `os`, `subprocess`, `open`,
    `eval`, `__import__`, dunder-attribute access, etc. are rejected pre-exec.
  - Runs in a subprocess with a wall-clock timeout so infinite loops / hangs
    in generated code can't stall a training run.
  - Still exec-based. If you scale this beyond a local research loop, run it
    in a real container/gVisor sandbox, not just AST filtering.
"""
from __future__ import annotations

import ast
import multiprocessing as mp
import re
import traceback
from typing import Optional

import numpy as np

FORBIDDEN_NAMES = {
    "os", "sys", "subprocess", "socket", "shutil", "pathlib", "importlib",
    "eval", "exec", "compile", "open", "__import__", "input", "exit",
    "quit", "globals", "locals", "vars", "breakpoint",
}
ALLOWED_IMPORTS = {"numpy", "re", "math", "string", "itertools", "collections"}


class UnsafeProgramError(Exception):
    pass


def static_check(code: str) -> None:
    """Raise UnsafeProgramError if the code references anything off the
    allowlist. Run this BEFORE exec, not instead of the subprocess timeout.
    """
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        raise UnsafeProgramError(f"SyntaxError: {e}")

    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            mod = node.module if isinstance(node, ast.ImportFrom) else None
            names = [mod] if mod else [a.name for a in node.names]
            for n in names:
                root = (n or "").split(".")[0]
                if root not in ALLOWED_IMPORTS:
                    raise UnsafeProgramError(f"Disallowed import: {n}")
        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            raise UnsafeProgramError(f"Disallowed name: {node.id}")
        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            raise UnsafeProgramError(f"Disallowed dunder attribute: {node.attr}")

    if "predict_attention" not in code:
        raise UnsafeProgramError("Program must define predict_attention(tokens)")


def _restricted_import(name, globals=None, locals=None, fromlist=(), level=0):
    root = name.split(".")[0]
    if root not in ALLOWED_IMPORTS:
        raise ImportError(f"import of {name!r} is not allowed in this sandbox")
    import importlib
    return importlib.import_module(name)


def _worker(code: str, tokens: list, out_queue: mp.Queue) -> None:
    try:
        static_check(code)
        safe_globals = {
            "__builtins__": {
                "range": range, "len": len, "enumerate": enumerate,
                "min": min, "max": max, "sum": sum, "abs": abs,
                "float": float, "int": int, "str": str, "bool": bool,
                "list": list, "dict": dict, "set": set, "tuple": tuple,
                "sorted": sorted, "zip": zip, "reversed": reversed,
                "isinstance": isinstance, "print": lambda *a, **k: None,
                "__import__": _restricted_import,
            },
            "np": np,
            "numpy": np,
            "re": re,
        }
        local_ns: dict = {}
        exec(code, safe_globals, local_ns)
        fn = local_ns.get("predict_attention")
        if fn is None:
            out_queue.put(("error", "predict_attention not defined"))
            return
        result = fn(tokens)
        result = np.asarray(result, dtype=np.float64)
        n = len(tokens)
        if result.shape != (n, n):
            out_queue.put(("error", f"bad output shape {result.shape}, expected {(n, n)}"))
            return
        if not np.all(np.isfinite(result)):
            out_queue.put(("error", "non-finite values in output"))
            return
        out_queue.put(("ok", result))
    except UnsafeProgramError as e:
        out_queue.put(("error", f"unsafe: {e}"))
    except Exception:
        out_queue.put(("error", traceback.format_exc(limit=2)))


def _safe_globals() -> dict:
    return {
        "__builtins__": {
            "range": range, "len": len, "enumerate": enumerate,
            "min": min, "max": max, "sum": sum, "abs": abs,
            "float": float, "int": int, "str": str, "bool": bool,
            "list": list, "dict": dict, "set": set, "tuple": tuple,
            "sorted": sorted, "zip": zip, "reversed": reversed,
            "isinstance": isinstance, "print": lambda *a, **k: None,
            "__import__": _restricted_import,
        },
        "np": np,
        "numpy": np,
        "re": re,
    }


def _postprocess(result, n: int) -> tuple[Optional[np.ndarray], bool, Optional[str]]:
    result = np.asarray(result, dtype=np.float64)
    if result.shape != (n, n):
        return None, False, f"bad output shape {result.shape}, expected {(n, n)}"
    if not np.all(np.isfinite(result)):
        return None, False, "non-finite values in output"
    row_sums = result.sum(axis=1, keepdims=True)
    if np.any(np.abs(row_sums - 1.0) > 1e-3):
        safe_sums = np.where(row_sums < 1e-10, 1.0, row_sums)
        result = result / safe_sums
    return result, True, None


def compile_program(code: str):
    """Static-check `code` and exec it IN-PROCESS, returning its
    `predict_attention` callable (or raising UnsafeProgramError).

    Fast path for the RL training loop over *trusted*, template-generated
    programs (see grpo/rewarding.py): paying a fresh `spawn` subprocess per
    candidate -- the isolation `run_program` gives untrusted policy-LM output
    -- would otherwise dominate wall-clock. Do NOT feed untrusted code here.
    """
    static_check(code)
    local_ns: dict = {}
    exec(code, _safe_globals(), local_ns)
    fn = local_ns.get("predict_attention")
    if fn is None:
        raise UnsafeProgramError("predict_attention not defined")
    return fn


def run_program_inproc(code: str, tokens: list[str]
                       ) -> tuple[Optional[np.ndarray], bool, Optional[str]]:
    """In-process (no subprocess, no timeout) counterpart to `run_program`,
    for trusted code only. Same (matrix, executable, error) contract."""
    try:
        fn = compile_program(code)
        return _postprocess(fn(tokens), len(tokens))
    except UnsafeProgramError as e:
        return None, False, f"unsafe: {e}"
    except Exception:
        return None, False, traceback.format_exc(limit=2)


def run_program(code: str, tokens: list[str], timeout: float = 5.0
                ) -> tuple[Optional[np.ndarray], bool, Optional[str]]:
    """Execute `code` against `tokens` in a subprocess with a hard timeout.

    Returns (attention_matrix_or_None, executable_flag, error_or_None).
    """
    ctx = mp.get_context("spawn")
    q: mp.Queue = ctx.Queue()
    p = ctx.Process(target=_worker, args=(code, tokens, q))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return None, False, f"timeout after {timeout}s"

    if q.empty():
        return None, False, "worker died without result (likely crashed/OOM)"

    status, payload = q.get()
    if status == "ok":
        row_sums = payload.sum(axis=1, keepdims=True)
        needs_norm = np.any(np.abs(row_sums - 1.0) > 1e-3)
        if needs_norm:
            safe_sums = np.where(row_sums < 1e-10, 1.0, row_sums)
            payload = payload / safe_sums
        return payload, True, None
    return None, False, payload


In [ ]:
%%writefile env/data.py
"""
Attention-example data structures.

`load_dataset` reads the schema documented below, which you should point at
whatever your `scripts/extract_attention_maps.py` run produces for real
model heads. `synthetic_head_dataset` needs no model weights or network
access at all -- it fabricates known-ground-truth heads (previous-token,
first-token, sentence-boundary, diagonal) so you can unit-test the env,
reward shaping, and a synthesis agent's prompting before spending any API
budget or GPU time on the real thing. Treat it as a fixture, not a
replacement for real extracted attention maps.

Expected on-disk schema (one JSON object per line):
    {
      "head_id": "gpt2:L0H5",
      "tokens": ["And", " so", ",", " the", ...],
      "attention": [[...], [...], ...]   # n x n, row-stochastic
    }
Adjust `load_dataset` if the upstream repo's notebooks serialize attention
maps differently (e.g. .npz per head, or a HF Dataset) -- I couldn't fetch
that repo's directory contents at build time to confirm the exact format;
this is deliberately kept as the single seam to edit.
"""
from __future__ import annotations

import dataclasses
import json
import random
from pathlib import Path

import numpy as np


@dataclasses.dataclass
class AttentionExample:
    head_id: str
    tokens: list[str]
    attention: np.ndarray  # (n, n), row-stochastic


def load_dataset(path: str | Path) -> list[AttentionExample]:
    path = Path(path)
    examples = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            examples.append(AttentionExample(
                head_id=obj["head_id"],
                tokens=obj["tokens"],
                attention=np.asarray(obj["attention"], dtype=np.float64),
            ))
    return examples


def group_by_head(examples: list[AttentionExample]) -> dict[str, list[AttentionExample]]:
    """extract_attention_maps.py writes one flat JSONL spanning every head in
    a model; AttentionProgramEnv expects examples from a single head. Group
    once after loading, then build one env per head_id you care about.
    """
    grouped: dict[str, list[AttentionExample]] = {}
    for e in examples:
        grouped.setdefault(e.head_id, []).append(e)
    return grouped


def split_train_val(examples: list[AttentionExample], val_frac: float = 0.2,
                     seed: int = 0) -> tuple[list[AttentionExample], list[AttentionExample]]:
    rng = random.Random(seed)
    idx = list(range(len(examples)))
    rng.shuffle(idx)
    n_val = max(1, int(len(examples) * val_frac))
    val_idx = set(idx[:n_val])
    train = [e for i, e in enumerate(examples) if i not in val_idx]
    val = [e for i, e in enumerate(examples) if i in val_idx]
    return train, val


# --------------------------------------------------------------------------
# Synthetic archetypes for testing (no model / network required)
# --------------------------------------------------------------------------

_SENTENCES = [
    "And so the daughter played on the slide and had a great time .",
    "Tim was happy because his mom gave him a new toy .",
    "The dog ran fast but it could not catch the cat .",
    "She asked her mom for a paper and a pen .",
    "Once upon a time there was a small red house .",
    "He looked at the sky and saw a bright star .",
]

# Multi-sentence inputs. On a single sentence, "attend to the sentence start"
# and "attend to token 0" are indistinguishable (both land on index 0) -- the
# degeneracy tests/test_env.py::_multi_sentence_example calls out. Passing
# multi_sentence=True to synthetic_head_dataset uses these instead so the
# sentence_boundary archetype is genuinely separable from first_token, which
# matters when training one policy to discriminate archetypes by their
# attention (see grpo/toy_data.py).
_MULTI_SENTENCES = [
    "Tim was happy . He ran outside and played with his dog .",
    "The dog barked loudly . A cat ran away fast . It was gone now .",
    "She smiled at him . Then she left the room . Nobody saw her go .",
    "Rain fell all day . The river rose high . People left their homes .",
    "He opened the box . Inside was a ring . His hands began to shake .",
    "Birds flew south . Winter came early . Snow covered the whole town .",
]


def _tokenize(sentence: str) -> list[str]:
    return sentence.split(" ")


def _row_softmax(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)


def _previous_token_attention(tokens: list[str]) -> np.ndarray:
    n = len(tokens)
    logits = np.full((n, n), -5.0)
    for i in range(n):
        logits[i, max(i - 1, 0)] = 5.0
        logits[i, i] = 1.0
    return _row_softmax(logits)


def _first_token_attention(tokens: list[str]) -> np.ndarray:
    n = len(tokens)
    logits = np.full((n, n), -5.0)
    logits[:, 0] = 5.0
    return _row_softmax(logits)


def _sentence_boundary_attention(tokens: list[str]) -> np.ndarray:
    n = len(tokens)
    logits = np.full((n, n), -5.0)
    for i in range(n):
        for j in range(i + 1):
            is_sent_start = (j == 0) or (tokens[j - 1] in {".", "!", "?"})
            if is_sent_start and tokens[j] not in {".", "!", "?", ","}:
                logits[i, j] = 3.0
        logits[i, i] = 0.5
    return _row_softmax(logits)


def _diagonal_self_attention(tokens: list[str]) -> np.ndarray:
    n = len(tokens)
    logits = np.eye(n) * 6.0 - 5.0
    return _row_softmax(logits)


ARCHETYPES = {
    "previous_token": _previous_token_attention,
    "first_token": _first_token_attention,
    "sentence_boundary": _sentence_boundary_attention,
    "diagonal_self": _diagonal_self_attention,
}


def synthetic_head_dataset(kind: str, n_examples: int = 20, seed: int = 0,
                            noise: float = 0.02,
                            multi_sentence: bool = False) -> list[AttentionExample]:
    """Generate a fake but internally-consistent dataset for one head
    archetype, useful as ground truth to sanity-check the env/reward before
    touching real extracted attention maps.

    multi_sentence: draw from multi-sentence inputs so archetypes that only
        diverge across sentence boundaries (sentence_boundary vs first_token)
        are separable -- used by the toy GRPO convergence demo.
    """
    if kind not in ARCHETYPES:
        raise ValueError(f"unknown archetype {kind!r}, choose from {list(ARCHETYPES)}")
    fn = ARCHETYPES[kind]
    corpus = _MULTI_SENTENCES if multi_sentence else _SENTENCES
    rng = np.random.default_rng(seed)
    examples = []
    for i in range(n_examples):
        sentence = corpus[i % len(corpus)]
        tokens = _tokenize(sentence)
        attn = fn(tokens)
        if noise > 0:
            attn = attn + rng.normal(0, noise, attn.shape)
            attn = np.clip(attn, 1e-6, None)
            attn = attn / attn.sum(axis=1, keepdims=True)
        examples.append(AttentionExample(head_id=f"synthetic:{kind}", tokens=tokens, attention=attn))
    return examples


In [ ]:
%%writefile env/attention_env.py
"""
AttentionProgramEnv: an OpenEnv-style RL environment where the policy
writes Python programs that approximate a target attention head, and the
environment scores them with the paper's own metrics (IoU / JSD) plus a
complexity penalty, with verifiable, non-LM-judged rewards.

This directly targets the paper's stated gap: they use one-shot generation
plus a single refinement round and explicitly say closing the fit gap
"likely requires richer synthesis strategies such as multi-round refinement
with stronger feedback signals." This env generalizes their fixed
one-round refinement into an N-round MDP so a policy can be trained with
GRPO (or any group-relative RL method) instead of relying on a single
prompted refinement pass.

API mirrors OpenEnv's Environment / Observation / StepResult conventions:
    obs = env.reset()
    ...
    step = env.step(Action(code=candidate_program))
    step.observation, step.reward, step.done, step.info
"""
from __future__ import annotations

import dataclasses
from typing import Optional

import numpy as np

from .data import AttentionExample
from .executor import run_program
from .reward import RewardBreakdown, compute_reward, positional_collapse_score


@dataclasses.dataclass
class Observation:
    head_id: str
    tokens: list[str]
    round_idx: int
    max_rounds: int
    # Feedback from the previous round, mirroring the paper's refinement
    # step: "identify representative best and worst-scoring examples ...
    # constructing structured error feedback by contrasting real and
    # predicted attention patterns". None on round 0.
    feedback: Optional[dict] = None
    # NOTE: target attention is intentionally NOT included here. The policy
    # only ever sees `tokens` (matching the paper's X -> A signature) plus
    # the top-k edge summary used to build the synthesis prompt upstream
    # (see scripts/build_prompt.py). Held-out attention is only used by the
    # env internally to score submissions.


@dataclasses.dataclass
class Action:
    code: str


@dataclasses.dataclass
class StepResult:
    observation: Optional[Observation]
    reward: float
    done: bool
    info: dict


class AttentionProgramEnv:
    def __init__(self, examples: list[AttentionExample], max_rounds: int = 3,
                 reward_weights: Optional[dict] = None, timeout: float = 5.0,
                 val_fraction: float = 0.3, seed: int = 0):
        """
        examples: all examples for ONE head (same head_id), used as both the
            fitting set (shown to the policy across rounds) and the held-out
            set (used only for scoring, never revealed as raw numbers).
        max_rounds: generalizes the paper's fixed single refinement round
            into a configurable budget.
        val_fraction: fraction of `examples` withheld purely for scoring,
            so a policy can't just memorize per-example feedback and must
            infer the underlying rule -- this matches the paper's own
            train/held-out split for selecting pi*.
        """
        if len({e.head_id for e in examples}) != 1:
            raise ValueError("AttentionProgramEnv expects examples from a single head")
        self.head_id = examples[0].head_id
        self.max_rounds = max_rounds
        self.reward_weights = reward_weights
        self.timeout = timeout

        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(examples))
        n_val = max(1, int(len(examples) * val_fraction))
        val_idx = set(idx[:n_val].tolist())
        self.fit_examples = [e for i, e in enumerate(examples) if i not in val_idx]
        self.val_examples = [e for i, e in enumerate(examples) if i in val_idx]
        if not self.fit_examples:
            self.fit_examples = examples
        if not self.val_examples:
            self.val_examples = examples

        self._round = 0
        self._current_example: Optional[AttentionExample] = None
        self._best_reward = -float("inf")
        self._best_code: Optional[str] = None

    def reset(self, example: Optional[AttentionExample] = None) -> Observation:
        self._round = 0
        self._current_example = example or self.fit_examples[
            np.random.randint(len(self.fit_examples))
        ]
        self._best_reward = -float("inf")
        self._best_code = None
        return Observation(
            head_id=self.head_id,
            tokens=self._current_example.tokens,
            round_idx=0,
            max_rounds=self.max_rounds,
            feedback=None,
        )

    def step(self, action: Action) -> StepResult:
        if self._current_example is None:
            raise RuntimeError("call reset() before step()")

        ex = self._current_example
        Ahat, executable, error = run_program(action.code, ex.tokens, timeout=self.timeout)
        collapse = positional_collapse_score(Ahat) if executable else 0.0
        breakdown: RewardBreakdown = compute_reward(
            ex.attention, Ahat, action.code, executable, error, self.reward_weights,
            collapse_score=collapse,
        )

        if breakdown.reward > self._best_reward:
            self._best_reward = breakdown.reward
            self._best_code = action.code

        self._round += 1
        done = self._round >= self.max_rounds

        feedback = None
        if not done:
            feedback = self._build_feedback(ex, Ahat, breakdown)

        obs = None
        if not done:
            obs = Observation(
                head_id=self.head_id,
                tokens=ex.tokens,
                round_idx=self._round,
                max_rounds=self.max_rounds,
                feedback=feedback,
            )

        info = {
            "iou": breakdown.iou,
            "jsd": breakdown.jsd,
            "complexity_penalty": breakdown.complexity_penalty,
            "executable": breakdown.executable,
            "error": breakdown.error,
            "best_reward_so_far": self._best_reward,
        }
        return StepResult(observation=obs, reward=breakdown.reward, done=done, info=info)

    def evaluate_held_out(self, code: str) -> dict:
        """Score `code` on the held-out split. This is what should decide
        pi* across a group of GRPO rollouts, not the in-loop training
        reward, to avoid rewarding overfit to the fit examples shown during
        refinement rounds.
        """
        ious, jsds = [], []
        n_fail = 0
        for ex in self.val_examples:
            Ahat, executable, error = run_program(code, ex.tokens, timeout=self.timeout)
            if not executable:
                n_fail += 1
                continue
            r = compute_reward(ex.attention, Ahat, code, executable, error, self.reward_weights)
            ious.append(r.iou)
            jsds.append(r.jsd)
        return {
            "mean_iou": float(np.mean(ious)) if ious else 0.0,
            "mean_jsd": float(np.mean(jsds)) if jsds else float(np.log(2.0)),
            "fail_rate": n_fail / max(len(self.val_examples), 1),
            "n_scored": len(ious),
        }

    @staticmethod
    def _build_feedback(ex: AttentionExample, Ahat: Optional[np.ndarray],
                         breakdown: RewardBreakdown, top_k: int = 6) -> dict:
        """Structured error feedback contrasting real vs. predicted attention
        on the worst-fit token pairs, mirroring the paper's refinement step
        (section 2.2). If the program failed to execute, feedback is just
        the error message -- that alone is often enough signal to fix a
        shape/naming bug on the next round.
        """
        if Ahat is None:
            return {"status": "execution_failed", "error": breakdown.error}

        diff = np.abs(ex.attention - Ahat)
        n = diff.shape[0]
        flat_idx = np.argsort(diff, axis=None)[::-1][:top_k]
        worst_edges = []
        for fi in flat_idx:
            i, j = np.unravel_index(fi, diff.shape)
            worst_edges.append({
                "query": ex.tokens[i], "query_idx": int(i),
                "key": ex.tokens[j], "key_idx": int(j),
                "real": round(float(ex.attention[i, j]), 4),
                "predicted": round(float(Ahat[i, j]), 4),
            })
        return {
            "status": "scored",
            "iou": round(breakdown.iou, 4),
            "jsd": round(breakdown.jsd, 4),
            "complexity_penalty": round(breakdown.complexity_penalty, 4),
            "worst_edges": worst_edges,
        }


In [ ]:
%%writefile grpo/__init__.py
"""From-scratch PyTorch GRPO for attention-program synthesis.

Public surface:
  - core:      GRPOConfig, Rollout, train_grpo, grpo_loss, compute_group_advantages
  - rewarding: ProgramScorer (verifiable IoU/JSD/degeneracy reward)
  - toy_policy: ToyProgramPolicy (self-contained DSL policy, no GPU/download)
  - train:     run_toy_training, make_reward_fn, HeadPrompt

The HF causal-LM policy lives in grpo.hf_policy and is imported lazily by
train_grpo_torch.py so `import grpo` never requires transformers.
"""
from grpo.core import (
    GRPOConfig, Policy, Rollout, compute_group_advantages, grpo_loss, train_grpo,
)
from grpo.rewarding import ProgramScorer
from grpo.train import HeadPrompt, make_reward_fn, run_toy_training

__all__ = [
    "GRPOConfig", "Policy", "Rollout", "compute_group_advantages", "grpo_loss",
    "train_grpo", "ProgramScorer", "HeadPrompt", "make_reward_fn", "run_toy_training",
]


In [ ]:
%%writefile grpo/core.py
"""
GRPO (Group Relative Policy Optimization) from scratch in PyTorch.

This is the "not scaffolded" core: the algorithm of Shao et al. (2024,
DeepSeekMath) implemented directly, not delegated to a library trainer. It
is deliberately backend-agnostic -- it knows nothing about attention heads,
programs, tokenizers, or transformers. It talks to a `Policy` (below) that
can (a) sample grouped rollouts and (b) recompute per-token log-probs and a
KL-to-reference term with gradient. Both the self-contained toy policy
(grpo/toy_policy.py) and the HF causal-LM policy (grpo/hf_policy.py) satisfy
that interface, so the exact same optimizer step trains both.

GRPO in one paragraph: for each prompt, sample a *group* of G completions;
score each with a verifiable reward; use the group's mean/std as a baseline
so the advantage of completion i is (r_i - mean)/std -- no learned value
network. Then take the standard PPO-style clipped surrogate on the
per-token importance ratio, plus a KL penalty toward a frozen reference
policy. Here the reward is the attention-program env's IoU/JSD/degeneracy
score (env/reward.py), which is verifiable and not LM-judged.
"""
from __future__ import annotations

import dataclasses
from typing import Callable, Protocol

import torch


@dataclasses.dataclass
class Rollout:
    """One sampled completion and everything needed to score and reweight it."""
    prompt_index: int          # which prompt/head this came from (groups share this)
    action: object             # backend-specific (toy: (template, sharpness); hf: token ids)
    text: str                  # the program string to execute for reward
    old_logps: torch.Tensor    # [T] per-token log-prob under the behavior policy (detached)
    reward: float = 0.0        # filled in by the driver via reward_fn


class Policy(Protocol):
    """The only surface grpo/core.py needs from a policy backend."""

    def sample(self, prompt_indices: list[int], group_size: int) -> list[Rollout]:
        """Return len(prompt_indices) * group_size rollouts, each tagged with
        its originating prompt_index. `old_logps` must be detached."""
        ...

    def evaluate(self, rollouts: list[Rollout]
                 ) -> tuple[list[torch.Tensor], list[torch.Tensor], list[torch.Tensor] | None]:
        """Recompute, WITH gradient, per-token log-probs, per-token KL to the
        frozen reference policy, and (optionally) per-token policy entropy, in
        the same token order as `old_logps`. Returns
        (new_logps, kl, entropies); entropies may be None if the backend
        does not provide an entropy bonus (e.g. large-vocab LM)."""
        ...

    def parameters(self):
        ...


@dataclasses.dataclass
class GRPOConfig:
    steps: int = 80
    group_size: int = 12         # G: completions sampled per prompt
    inner_epochs: int = 1        # PPO-style reuse of each batch of rollouts
    lr: float = 3e-3
    clip_eps: float = 0.2        # PPO clip range on the importance ratio
    kl_coef: float = 0.02        # weight on KL-to-reference
    ent_coef: float = 0.01       # entropy bonus; guards against exploration collapse
    adv_eps: float = 1e-4        # std floor when standardizing group advantages
    max_grad_norm: float = 1.0
    seed: int = 0


# --------------------------------------------------------------------------
# Core algorithm
# --------------------------------------------------------------------------

def compute_group_advantages(rewards: torch.Tensor, group_ids: list[int],
                             eps: float = 1e-4) -> torch.Tensor:
    """GRPO advantage: standardize each prompt-group's rewards to (r-mean)/std.

    The group mean is the value baseline (no critic); the group std is a
    per-prompt scale. Groups where every completion scored identically get
    zero advantage (nothing to learn from that prompt this step)."""
    adv = torch.zeros_like(rewards)
    groups: dict[int, list[int]] = {}
    for i, g in enumerate(group_ids):
        groups.setdefault(g, []).append(i)
    for idxs in groups.values():
        r = rewards[idxs]
        std = r.std(unbiased=False)
        adv[idxs] = (r - r.mean()) / (std + eps)
    return adv


def grpo_loss(new_logps: list[torch.Tensor], old_logps: list[torch.Tensor],
              advantages: torch.Tensor, kls: list[torch.Tensor],
              clip_eps: float, kl_coef: float,
              entropies: list[torch.Tensor] | None = None,
              ent_coef: float = 0.0) -> tuple[torch.Tensor, dict]:
    """Token-mean PPO-clipped surrogate + KL penalty - entropy bonus.

    For rollout i with scalar advantage A_i and per-token ratio
    r_t = exp(logp_new - logp_old):
        L_t = -min(r_t*A_i, clip(r_t, 1-eps, 1+eps)*A_i) + kl_coef*KL_t - ent_coef*H_t
    averaged over all tokens in the batch. Clipping caps how far one update
    can move the policy toward a high-advantage completion; the KL term
    anchors it to the frozen reference so it doesn't drift into reward hacking
    the degeneracy penalty; the entropy bonus keeps sampling diverse so the
    policy doesn't prematurely collapse onto a locally-good program before
    discovering a better one."""
    # Align everything to the policy's compute device: a GPU policy returns
    # new_logps/kls on cuda, while old_logps (cached in sample()) and the
    # advantages (built from CPU rewards) live on CPU.
    device = new_logps[0].device if new_logps else advantages.device
    advantages = advantages.to(device)
    pol_sum = torch.zeros((), device=device)
    kl_sum = torch.zeros((), device=device)
    ent_sum = torch.zeros((), device=device)
    n_tokens = 0
    for i, (lp_new, lp_old, adv_i, kl_i) in enumerate(zip(new_logps, old_logps, advantages, kls)):
        lp_old = lp_old.to(device)
        kl_i = kl_i.to(device)
        ratio = torch.exp(lp_new - lp_old)                 # [T]
        unclipped = ratio * adv_i
        clipped = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv_i
        pol_sum = pol_sum - torch.min(unclipped, clipped).sum()
        kl_sum = kl_sum + kl_i.sum()
        if entropies is not None and entropies[i] is not None:
            ent_sum = ent_sum + entropies[i].to(device).sum()
        n_tokens += lp_new.numel()
    n_tokens = max(n_tokens, 1)
    loss = (pol_sum + kl_coef * kl_sum - ent_coef * ent_sum) / n_tokens
    stats = {
        "policy_loss": float(pol_sum.detach() / n_tokens),
        "kl": float(kl_sum.detach() / n_tokens),
        "entropy": float(ent_sum.detach() / n_tokens),
    }
    return loss, stats


# --------------------------------------------------------------------------
# Training loop
# --------------------------------------------------------------------------

RewardFn = Callable[[int, str], float]


def train_grpo(policy: Policy, num_prompts: int, reward_fn: RewardFn,
               cfg: GRPOConfig, on_step: Callable[[dict], None] | None = None
               ) -> list[dict]:
    """Run GRPO for `cfg.steps` steps over `num_prompts` prompts.

    reward_fn(prompt_index, program_text) -> float is the verifiable reward
    (env/reward.py). on_step, if given, receives a per-step metrics dict --
    used by callers to print reward curves or track per-prompt best programs.
    Returns the list of per-step metrics dicts.
    """
    torch.manual_seed(cfg.seed)
    opt = torch.optim.Adam(policy.parameters(), lr=cfg.lr)
    prompt_indices = list(range(num_prompts))
    history: list[dict] = []

    for step in range(cfg.steps):
        rollouts = policy.sample(prompt_indices, cfg.group_size)
        for r in rollouts:
            r.reward = reward_fn(r.prompt_index, r.text)

        rewards = torch.tensor([r.reward for r in rollouts], dtype=torch.float32)
        group_ids = [r.prompt_index for r in rollouts]
        advantages = compute_group_advantages(rewards, group_ids, cfg.adv_eps)

        last_stats: dict = {}
        for _ in range(cfg.inner_epochs):
            new_logps, kls, entropies = policy.evaluate(rollouts)
            loss, last_stats = grpo_loss(
                new_logps, [r.old_logps for r in rollouts], advantages, kls,
                cfg.clip_eps, cfg.kl_coef, entropies, cfg.ent_coef,
            )
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(policy.parameters()), cfg.max_grad_norm)
            opt.step()

        metrics = {
            "step": step,
            "mean_reward": float(rewards.mean()),
            "max_reward": float(rewards.max()),
            "frac_executable": float((rewards > -1.0).float().mean()),
            "loss": float(loss.detach()),
            **last_stats,
        }
        history.append(metrics)
        if on_step is not None:
            on_step(metrics)
    return history


In [ ]:
%%writefile grpo/program_dsl.py
"""
A tiny program-synthesis action space for the *self-contained* GRPO backend.

The real research setting (scripts/grpo_train.py, grpo/hf_policy.py) has a
code LM emit arbitrary Python `predict_attention` functions. That needs a
GPU and a capable policy model to produce *executable* code at all, which
makes it a poor smoke test. This module gives a from-scratch PyTorch policy
(grpo/toy_policy.py) something it can actually learn end-to-end on a laptop
in seconds, while exercising the *exact same* GRPO machinery, verifiable
reward, sandboxed executor, and IoU/JSD metrics as the LM path.

An action is a length-2 token sequence `(template_idx, sharpness_idx)`:
  - template_idx picks one of a handful of parametric attention motifs
    (previous-token, first-token, diagonal/self, two-back, sentence-start),
  - sharpness_idx picks how peaked the softmax over target positions is.
`compile_action` turns that into a genuine Python program string in the same
`predict_attention(tokens) -> np.ndarray` contract the executor and reward
expect -- so a learned toy action is scored by identical code to a learned
LM completion. This mirrors the paper's own "program library" framing, but
the retrieval/composition policy is *learned by RL* instead of fixed.
"""
from __future__ import annotations

import numpy as np

# Sharpness = logit added at the target position(s) over a zero baseline on
# all other causal positions; larger -> more peaked row-softmax. Chosen to
# straddle the synthetic archetypes' own peak strength (~5-6 in env/data.py),
# so the policy has a real parameter to tune, not a no-op dimension.
SHARPNESS_CHOICES = (2.0, 4.0, 6.0)

# Each template is (name, body) where `body` is inserted inside a
# `for i in range(n):` loop and may set logits[i, j] += SCALE. `{scale}` is
# substituted with the chosen sharpness value at compile time.
_TEMPLATES: list[tuple[str, str]] = [
    ("previous_token",
     "        logits[i, max(i - 1, 0)] += {scale}"),
    ("first_token",
     "        logits[i, 0] += {scale}"),
    ("diagonal_self",
     "        logits[i, i] += {scale}"),
    ("two_back",
     "        logits[i, max(i - 2, 0)] += {scale}"),
    ("sentence_boundary",
     "        for j in range(i + 1):\n"
     "            is_start = (j == 0) or (tokens[j - 1] in ('.', '!', '?'))\n"
     "            if is_start and tokens[j] not in ('.', '!', '?', ','):\n"
     "                logits[i, j] += {scale}"),
]

TEMPLATE_NAMES: list[str] = [name for name, _ in _TEMPLATES]
NUM_TEMPLATES: int = len(_TEMPLATES)
NUM_SHARPNESS: int = len(SHARPNESS_CHOICES)

_PROGRAM_SKELETON = '''\
import numpy as np

def predict_attention(tokens):
    """{name} (sharpness={scale})."""
    n = len(tokens)
    logits = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if j > i:
                logits[i, j] = -1e9
    for i in range(n):
{body}
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)
'''


def compile_action(template_idx: int, sharpness_idx: int) -> str:
    """Map a (template_idx, sharpness_idx) action to a real Python program
    string in the `predict_attention(tokens)` contract."""
    name, body = _TEMPLATES[template_idx]
    scale = SHARPNESS_CHOICES[sharpness_idx]
    return _PROGRAM_SKELETON.format(name=name, scale=scale, body=body.format(scale=scale))


def action_str(template_idx: int, sharpness_idx: int) -> str:
    return f"{TEMPLATE_NAMES[template_idx]}@{SHARPNESS_CHOICES[sharpness_idx]:g}"


# --------------------------------------------------------------------------
# Policy-facing features
# --------------------------------------------------------------------------
# The LM policy is shown the top attention edges as text (scripts/build_prompt.py).
# The toy policy is shown a compact numeric summary of the *same* information:
# how much mass a head places on each positional motif. A head archetype maps
# to a distinctive feature vector, so a small MLP can learn template selection
# from reward alone. Kept deliberately motif-aligned (one feature per template
# family) rather than hand-mapping features to templates -- the mapping from
# features to the best action is what GRPO has to discover.

FEATURE_NAMES = [
    "mass_self", "mass_prev", "mass_two_back", "mass_first",
    "mass_sentence_start", "col_spread",
]
FEATURE_DIM = len(FEATURE_NAMES)


def attention_features(attention: np.ndarray, tokens: list[str]) -> np.ndarray:
    """Compact positional-motif summary of one attention matrix, in [0, 1]."""
    A = np.clip(np.asarray(attention, dtype=np.float64), 0.0, None)
    n = A.shape[0]
    rows = np.arange(n)
    mass_self = A[rows, rows].mean()
    mass_prev = A[rows, np.maximum(rows - 1, 0)].mean()
    mass_two_back = A[rows, np.maximum(rows - 2, 0)].mean()
    mass_first = A[:, 0].mean()

    is_start = np.zeros(n, dtype=bool)
    for j in range(n):
        is_start[j] = (j == 0) or (tokens[j - 1] in (".", "!", "?"))
    start_cols = np.where(is_start)[0]
    causal_start = np.zeros(n)
    for i in range(n):
        cols = start_cols[start_cols <= i]
        causal_start[i] = A[i, cols].sum() if len(cols) else 0.0
    mass_sentence_start = causal_start.mean()

    # Normalized entropy of the column marginal (the same behavioral signal
    # reward.positional_collapse_score uses). This is what separates a head
    # that dumps all mass on one column (first_token -> col_spread ~ 0) from
    # one that spreads across several sentence-start columns
    # (sentence_boundary -> col_spread well above 0), even though both put
    # high mass on token 0. Without it the two archetypes are near-degenerate
    # in feature space and the policy conflates them.
    col_marginal = A.mean(axis=0)
    total = col_marginal.sum()
    if total <= 1e-10 or n <= 1:
        col_spread = 0.0
    else:
        p = col_marginal / total
        entropy = -np.sum(p * np.log(p + 1e-12))
        col_spread = float(np.clip(entropy / np.log(n), 0.0, 1.0))

    return np.array([mass_self, mass_prev, mass_two_back, mass_first,
                     mass_sentence_start, col_spread], dtype=np.float32)


def mean_features(examples) -> np.ndarray:
    """Average feature vector over a head's examples (the policy input)."""
    feats = [attention_features(e.attention, e.tokens) for e in examples]
    return np.mean(feats, axis=0).astype(np.float32)


In [ ]:
%%writefile grpo/rewarding.py
"""
Turns a program string into a scalar GRPO reward, fast.

Wraps env/reward.py's paper-faithful IoU/JSD + degeneracy penalty. The only
thing added here is a compile cache: an RL step scores G-completions x
prompts x steps programs, and for the toy DSL those collapse to a handful of
distinct source strings, so we compile each once (in-process, trusted path)
and reuse the callable. The reward *value* is identical to what
AttentionProgramEnv.step would compute; this just skips the per-call spawn
subprocess that isolates untrusted policy-LM output.
"""
from __future__ import annotations

import numpy as np

from env.executor import compile_program, run_program, run_program_inproc
from env.reward import FAILURE_REWARD, compute_reward, positional_collapse_score


class ProgramScorer:
    def __init__(self, weights: dict | None = None, trusted: bool = True):
        """trusted=True executes candidates in-process (fast; use for the
        template-generated toy DSL). trusted=False routes through the
        subprocess sandbox with a timeout (use for untrusted policy-LM code).
        """
        self.weights = weights
        self.trusted = trusted
        self._fn_cache: dict = {}

    def _run(self, code: str, tokens: list[str]):
        if not self.trusted:
            return run_program(code, tokens)
        cached = self._fn_cache.get(code)
        if cached is None:
            try:
                cached = compile_program(code)
            except Exception as e:  # noqa: BLE001 - static-check / syntax failures
                cached = e
            self._fn_cache[code] = cached
        if isinstance(cached, Exception):
            return None, False, f"compile failed: {cached}"
        try:
            from env.executor import _postprocess
            return _postprocess(cached(tokens), len(tokens))
        except Exception as e:  # noqa: BLE001 - runtime failures in candidate
            return None, False, f"runtime error: {e}"

    def score_example(self, code: str, example) -> float:
        A_hat, executable, error = self._run(code, example.tokens)
        collapse = positional_collapse_score(A_hat) if executable else 0.0
        return compute_reward(example.attention, A_hat, code, executable, error,
                              self.weights, collapse_score=collapse).reward

    def mean_reward(self, code: str, examples: list) -> float:
        """Reward averaged over a head's examples -- a lower-variance RL
        signal than scoring a single sampled sentence, and it rewards
        programs that describe the head's *general* rule, not one input."""
        if not examples:
            return FAILURE_REWARD
        return float(np.mean([self.score_example(code, e) for e in examples]))


In [ ]:
%%writefile grpo/toy_policy.py
"""
A self-contained PyTorch policy over the program DSL (grpo/program_dsl.py).

This is what makes the repo *runnable end-to-end with no GPU and no model
download*: a small autoregressive network that, conditioned on a head's
attention-motif features, emits a two-token program `(template, sharpness)`.
Sampling, per-token log-probs, and an exact categorical KL to a frozen
reference are all real -- they feed the same grpo/core.py optimizer step the
HF causal-LM policy uses. Training this against the verifiable env reward
visibly converges (the policy learns which program each head archetype
wants), which is the point: it demonstrates the full RL loop working, not a
stub.

`ToyProgramPolicy` is a plain wrapper (not an nn.Module) holding a trainable
`ToyNet` and a frozen reference `ToyNet`, so `parameters()` never leaks the
reference into the optimizer.
"""
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from grpo.core import Rollout
from grpo.program_dsl import (
    FEATURE_DIM, NUM_SHARPNESS, NUM_TEMPLATES, compile_action,
)


class ToyNet(nn.Module):
    def __init__(self, feature_dim: int = FEATURE_DIM, hidden: int = 64,
                 template_emb: int = 16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.template_head = nn.Linear(hidden, NUM_TEMPLATES)
        self.template_emb = nn.Embedding(NUM_TEMPLATES, template_emb)
        self.sharp_head = nn.Linear(hidden + template_emb, NUM_SHARPNESS)

    def encode(self, feats: torch.Tensor) -> torch.Tensor:
        return self.encoder(feats)

    def template_logits(self, h: torch.Tensor) -> torch.Tensor:
        return self.template_head(h)

    def sharp_logits(self, h: torch.Tensor, template: torch.Tensor) -> torch.Tensor:
        return self.sharp_head(torch.cat([h, self.template_emb(template)], dim=-1))


def _step_kl(logits: torch.Tensor, ref_logits: torch.Tensor) -> torch.Tensor:
    """Exact per-example KL( current || reference ) for one categorical step."""
    p = F.softmax(logits, dim=-1)
    logp = F.log_softmax(logits, dim=-1)
    logq = F.log_softmax(ref_logits, dim=-1)
    return (p * (logp - logq)).sum(dim=-1)


class ToyProgramPolicy:
    """Policy protocol impl for the DSL backend."""

    def __init__(self, prompt_features: np.ndarray, hidden: int = 64,
                 device: str = "cpu", seed: int = 0):
        torch.manual_seed(seed)
        self.device = torch.device(device)
        self.features = torch.tensor(np.asarray(prompt_features, dtype=np.float32),
                                     device=self.device)  # [P, F]
        self.net = ToyNet(feature_dim=self.features.shape[1], hidden=hidden).to(self.device)
        self.ref = copy.deepcopy(self.net).to(self.device)
        for p in self.ref.parameters():
            p.requires_grad_(False)
        self.ref.eval()

    def parameters(self):
        return self.net.parameters()

    # -- sampling (no grad; produces detached behavior-policy log-probs) --
    def sample(self, prompt_indices: list[int], group_size: int) -> list[Rollout]:
        flat = [pi for pi in prompt_indices for _ in range(group_size)]
        idx = torch.tensor(flat, device=self.device, dtype=torch.long)
        with torch.no_grad():
            h = self.net.encode(self.features[idx])                 # [R, H]
            t_dist = torch.distributions.Categorical(logits=self.net.template_logits(h))
            t = t_dist.sample()                                     # [R]
            lp_t = t_dist.log_prob(t)
            s_dist = torch.distributions.Categorical(logits=self.net.sharp_logits(h, t))
            s = s_dist.sample()                                     # [R]
            lp_s = s_dist.log_prob(s)

        rollouts = []
        for k, pi in enumerate(flat):
            ti, si = int(t[k]), int(s[k])
            old_logps = torch.stack([lp_t[k], lp_s[k]]).detach().cpu()
            rollouts.append(Rollout(prompt_index=pi, action=(ti, si),
                                    text=compile_action(ti, si), old_logps=old_logps))
        return rollouts

    # -- re-evaluation (with grad; recomputes log-probs, KL, entropy) --
    def evaluate(self, rollouts: list[Rollout]):
        idx = torch.tensor([r.prompt_index for r in rollouts], device=self.device, dtype=torch.long)
        t = torch.tensor([r.action[0] for r in rollouts], device=self.device, dtype=torch.long)
        s = torch.tensor([r.action[1] for r in rollouts], device=self.device, dtype=torch.long)

        h = self.net.encode(self.features[idx])
        t_logits = self.net.template_logits(h)
        s_logits = self.net.sharp_logits(h, t)
        lp_t = F.log_softmax(t_logits, dim=-1).gather(1, t[:, None]).squeeze(1)
        lp_s = F.log_softmax(s_logits, dim=-1).gather(1, s[:, None]).squeeze(1)

        with torch.no_grad():
            h_ref = self.ref.encode(self.features[idx])
            t_logits_ref = self.ref.template_logits(h_ref)
            s_logits_ref = self.ref.sharp_logits(h_ref, t)
        kl_t = _step_kl(t_logits, t_logits_ref)
        kl_s = _step_kl(s_logits, s_logits_ref)

        ent_t = torch.distributions.Categorical(logits=t_logits).entropy()
        ent_s = torch.distributions.Categorical(logits=s_logits).entropy()

        n = len(rollouts)
        new_logps = [torch.stack([lp_t[k], lp_s[k]]) for k in range(n)]
        kls = [torch.stack([kl_t[k], kl_s[k]]) for k in range(n)]
        entropies = [torch.stack([ent_t[k], ent_s[k]]) for k in range(n)]
        return new_logps, kls, entropies

    @torch.no_grad()
    def greedy_actions(self) -> list[tuple[int, int]]:
        """Argmax (template, sharpness) per prompt -- for reporting what the
        policy has learned to pick for each head."""
        h = self.net.encode(self.features)
        t = self.net.template_logits(h).argmax(dim=-1)
        s = self.net.sharp_logits(h, t).argmax(dim=-1)
        return [(int(t[i]), int(s[i])) for i in range(self.features.shape[0])]


In [ ]:
%%writefile grpo/hf_policy.py
"""
HF causal-LM policy backend for the same from-scratch GRPO core.

This is the "real" research setting: the policy is a code LM that emits an
arbitrary Python `predict_attention` function as text, sampled with
`model.generate`. It implements the identical Policy protocol the toy DSL
policy does -- sample grouped rollouts, then recompute per-token log-probs
and a KL-to-reference term with gradient -- so grpo/core.py trains it with
the exact same optimizer step. The reward is still the verifiable env score
(untrusted code, so ProgramScorer routes it through the subprocess sandbox).

The per-token log-prob and KL math is written out by hand rather than pulled
from a trainer:
  - logp of completion token c_k comes from the logits at the *previous*
    position (P-1+k for a length-P prompt), gathered and log-softmaxed.
  - KL to the reference uses the k3 estimator kl = exp(d) - d - 1 with
    d = logp_ref - logp_new -- unbiased, non-negative, and cheap (no full
    vocab sum), which matters when the vocab is 50k+.

`CharTokenizer` + a locally-constructed tiny GPT-2 let the whole path run
offline (see tests/test_grpo_torch.py). In production you pass
`AutoTokenizer.from_pretrained(m)` and `AutoModelForCausalLM.from_pretrained(m)`.
"""
from __future__ import annotations

import copy
import re

import torch
import torch.nn.functional as F

from grpo.core import Rollout

_CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.DOTALL)


def extract_code(completion: str) -> str:
    """Pull a Python code block from an LM completion; fall back to raw text
    (some models emit the bare function when told 'respond with ONLY ...')."""
    m = _CODE_BLOCK_RE.search(completion)
    return (m.group(1) if m else completion).strip()


def _is_dispatched(model) -> bool:
    """True if accelerate has already placed the model across device(s)
    (device_map=...); in that case we must not call .to(device) on it."""
    return getattr(model, "hf_device_map", None) is not None


def _sequence_logps(model, prompt_ids: torch.Tensor, completion_ids: torch.Tensor) -> torch.Tensor:
    """Per-token log-prob of `completion_ids` continuing `prompt_ids`.

    logits at position t predict token t+1, so the k-th completion token
    (absolute position P+k) is predicted by logits[P-1+k]. log_softmax is
    taken in float32 so bf16/fp16 policies stay numerically stable."""
    ids = torch.cat([prompt_ids, completion_ids])[None]     # [1, P+T]
    logits = model(ids).logits[0]                           # [P+T, V]
    P, T = prompt_ids.shape[0], completion_ids.shape[0]
    pred = logits[P - 1: P - 1 + T].float()                # [T, V]
    return F.log_softmax(pred, dim=-1).gather(1, completion_ids[:, None]).squeeze(1)


class HFPolicy:
    def __init__(self, model, tokenizer, prompts: list[str], ref_model=None,
                 device: str = "cpu", max_new_tokens: int = 256,
                 temperature: float = 1.0, top_p: float = 1.0,
                 add_special_tokens: bool = True):
        # device_map-loaded / accelerate models place themselves; only .to()
        # when the caller passes a plain device string for a single-device model.
        if _is_dispatched(model):
            self.model = model
            self.device = next(model.parameters()).device
        else:
            self.device = torch.device(device)
            self.model = model.to(self.device)
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        # add_special_tokens=False when `prompts` are already chat-templated
        # strings (they carry their own special tokens as text).
        self.add_special_tokens = add_special_tokens

        # Frozen reference for the KL anchor.
        self._peft_ref = False
        if ref_model is not None:
            self.ref = ref_model if _is_dispatched(ref_model) else ref_model.to(self.device)
            for p in self.ref.parameters():
                p.requires_grad_(False)
            self.ref.eval()
        elif hasattr(model, "disable_adapter") and callable(model.disable_adapter):
            # PEFT/LoRA: the reference is just the base model with adapters
            # switched off -- no second full copy of the weights in memory.
            # (LoRA is zero-initialized, so ref == policy at step 0: KL 0.)
            self.ref = None
            self._peft_ref = True
        else:
            self.ref = copy.deepcopy(model).to(self.device)
            for p in self.ref.parameters():
                p.requires_grad_(False)
            self.ref.eval()
        # Keep the policy in eval mode throughout (dropout off) so log-probs
        # are a deterministic function of the params: the behavior-policy
        # `old_logps` and the re-evaluated `new_logps` then agree exactly at
        # inner-epoch 0, making the importance ratio 1 as GRPO assumes.
        # eval() only disables dropout/batchnorm; gradients still flow.
        self.model.eval()

        self.eos_id = getattr(tokenizer, "eos_token_id", None)
        self.pad_id = getattr(tokenizer, "pad_token_id", None) or self.eos_id

    def parameters(self):
        # Only the params that actually train (all of them for full FT; just
        # the adapters for LoRA) -- so the optimizer never allocates state for
        # a frozen 4B base model.
        return [p for p in self.model.parameters() if p.requires_grad]

    def _encode(self, text: str) -> torch.Tensor:
        ids = self.tokenizer.encode(text, add_special_tokens=self.add_special_tokens)
        return torch.tensor(ids, dtype=torch.long, device=self.device)

    def _truncate_completion(self, comp_ids: torch.Tensor) -> torch.Tensor:
        """Keep completion tokens up to and including the first eos; drop the
        trailing padding `generate` emits so log-probs aren't computed over pad."""
        if self.eos_id is not None:
            hit = (comp_ids == self.eos_id).nonzero()
            if hit.numel() > 0:
                return comp_ids[: int(hit[0]) + 1]
        return comp_ids

    def sample(self, prompt_indices: list[int], group_size: int) -> list[Rollout]:
        rollouts: list[Rollout] = []
        self.model.eval()
        for pi in prompt_indices:
            prompt_ids = self._encode(self.prompts[pi])
            with torch.no_grad():
                gen = self.model.generate(
                    prompt_ids[None],
                    attention_mask=torch.ones_like(prompt_ids)[None],  # silence pad==eos warning
                    do_sample=True, num_return_sequences=group_size,
                    max_new_tokens=self.max_new_tokens, temperature=self.temperature,
                    top_p=self.top_p, pad_token_id=self.pad_id,
                )
            P = prompt_ids.shape[0]
            for g in range(group_size):
                comp_ids = self._truncate_completion(gen[g, P:])
                if comp_ids.numel() == 0:
                    comp_ids = gen[g, P:P + 1]  # guard: never zero-length
                with torch.no_grad():
                    old_lp = _sequence_logps(self.model, prompt_ids, comp_ids).detach().cpu()
                # skip_special_tokens=True: instruct models end a turn with an
                # eos like <|im_end|>; left in, it glues onto `return attention`
                # and the program fails to parse.
                decoded = self.tokenizer.decode(comp_ids.tolist(), skip_special_tokens=True)
                text = extract_code(decoded)
                rollouts.append(Rollout(
                    prompt_index=pi, action=(prompt_ids.cpu(), comp_ids.cpu()),
                    text=text, old_logps=old_lp,
                ))
        return rollouts

    def _ref_logps(self, prompt_ids: torch.Tensor, comp_ids: torch.Tensor) -> torch.Tensor:
        if self._peft_ref:
            # Reference = base model with LoRA adapters disabled.
            with torch.no_grad(), self.model.disable_adapter():
                return _sequence_logps(self.model, prompt_ids, comp_ids)
        with torch.no_grad():
            return _sequence_logps(self.ref, prompt_ids, comp_ids)

    def evaluate(self, rollouts: list[Rollout]):
        new_logps, kls = [], []
        for r in rollouts:
            prompt_ids = r.action[0].to(self.device)
            comp_ids = r.action[1].to(self.device)
            lp_new = _sequence_logps(self.model, prompt_ids, comp_ids)         # grad
            lp_ref = self._ref_logps(prompt_ids, comp_ids)                     # frozen
            d = lp_ref - lp_new
            kl = torch.exp(d) - d - 1.0                                        # k3 estimator
            new_logps.append(lp_new)
            kls.append(kl)
        return new_logps, kls, None  # no entropy bonus for large-vocab LM


# --------------------------------------------------------------------------
# Offline test aid: a trivial char tokenizer so the HF path runs with a
# locally-constructed tiny model and zero network access.
# --------------------------------------------------------------------------

class CharTokenizer:
    """Byte-ish char tokenizer implementing the .encode/.decode/.eos_token_id/
    .pad_token_id surface HFPolicy needs. Real runs use AutoTokenizer."""
    eos_token_id = 256
    pad_token_id = 257
    vocab_size = 258

    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        return [min(ord(c), 255) for c in text]

    def decode(self, ids: list[int], skip_special_tokens: bool = True) -> str:
        return "".join(chr(i) for i in ids if i < 256)


In [ ]:
%%writefile grpo/train.py
"""
Orchestration glue shared by the CLI (train_grpo_torch.py) and the tests:
build the reward function from a set of head-prompts, and run a full toy
GRPO training session end to end.

Kept backend-agnostic where it can be: `make_reward_fn` and `HeadPrompt`
know nothing about the policy. `run_toy_training` wires the DSL policy,
verifiable reward, and grpo/core.py together and reports what the policy
learned to pick for each head.
"""
from __future__ import annotations

import dataclasses
from typing import Callable

import numpy as np

from grpo.core import GRPOConfig, train_grpo
from grpo.rewarding import ProgramScorer


@dataclasses.dataclass
class HeadPrompt:
    head_id: str
    examples: list                       # examples used to compute the reward
    expected_template: int | None = None  # ground-truth template idx (toy eval only)


def make_reward_fn(prompts: list[HeadPrompt], scorer: ProgramScorer
                   ) -> Callable[[int, str], float]:
    def reward_fn(prompt_index: int, code: str) -> float:
        return scorer.mean_reward(code, prompts[prompt_index].examples)
    return reward_fn


def run_toy_training(archetypes: list[str] | None = None,
                     cfg: GRPOConfig | None = None,
                     n_examples: int = 6, hidden: int = 64,
                     multi_sentence: bool = True, device: str = "cpu",
                     verbose: bool = True) -> dict:
    """Train one DSL policy to pick the right program for each head archetype.

    Returns a dict with the metrics history, the trained policy, the prompts,
    the policy's final greedy action per head, and template-selection accuracy
    (fraction of heads for which the argmax template matches ground truth).
    """
    # Lazy import avoids a module-level cycle (toy_data imports HeadPrompt).
    from grpo.program_dsl import TEMPLATE_NAMES, action_str, mean_features
    from grpo.toy_data import build_toy_prompts
    from grpo.toy_policy import ToyProgramPolicy

    cfg = cfg or GRPOConfig()
    prompts = build_toy_prompts(archetypes, n_examples=n_examples,
                                multi_sentence=multi_sentence, seed=cfg.seed)
    features = np.stack([mean_features(p.examples) for p in prompts])

    policy = ToyProgramPolicy(features, hidden=hidden, device=device, seed=cfg.seed)
    scorer = ProgramScorer(trusted=True)
    reward_fn = make_reward_fn(prompts, scorer)

    def _log(m: dict) -> None:
        if verbose and (m["step"] % max(cfg.steps // 10, 1) == 0 or m["step"] == cfg.steps - 1):
            print(f"  step {m['step']:3d} | mean_reward {m['mean_reward']:+.3f} "
                  f"| max {m['max_reward']:+.3f} | exec {m['frac_executable']:.2f} "
                  f"| kl {m.get('kl', 0):.4f}")

    history = train_grpo(policy, num_prompts=len(prompts), reward_fn=reward_fn,
                         cfg=cfg, on_step=_log)

    greedy = policy.greedy_actions()
    correct = 0
    report = []
    for p, (t, s) in zip(prompts, greedy):
        ok = (p.expected_template is None) or (t == p.expected_template)
        correct += int(ok)
        report.append({
            "head_id": p.head_id, "chose": action_str(t, s),
            "expected": TEMPLATE_NAMES[p.expected_template] if p.expected_template is not None else None,
            "reward": reward_fn(prompts.index(p), _compiled(t, s)),
            "correct": ok,
        })
    accuracy = correct / len(prompts)
    if verbose:
        print(f"\n  template-selection accuracy: {accuracy:.0%} "
              f"({correct}/{len(prompts)} heads)")
        for r in report:
            mark = "OK " if r["correct"] else "XX "
            print(f"    {mark}{r['head_id']:28s} chose {r['chose']:22s} "
                  f"(reward {r['reward']:+.3f})")

    return {"history": history, "policy": policy, "prompts": prompts,
            "greedy": greedy, "accuracy": accuracy, "report": report}


def _compiled(t: int, s: int) -> str:
    from grpo.program_dsl import compile_action
    return compile_action(t, s)


In [ ]:
%%writefile grpo/toy_data.py
"""
Builds the toy GRPO training problem: one head per attention archetype, each
with known ground truth, so we can both compute a verifiable reward and check
whether the learned policy selects the correct program template.

Archetype names in env/data.py line up 1:1 with template names in
grpo/program_dsl.py (previous_token, first_token, diagonal_self,
sentence_boundary), so the expected template index is just the template whose
name matches the archetype. two_back exists as a template but has no matching
archetype -- it is a distractor action the policy must learn *not* to pick.
"""
from __future__ import annotations

from env.data import synthetic_head_dataset
from grpo.program_dsl import TEMPLATE_NAMES
from grpo.train import HeadPrompt

DEFAULT_ARCHETYPES = ["previous_token", "first_token", "diagonal_self", "sentence_boundary"]


def build_toy_prompts(archetypes: list[str] | None = None, n_examples: int = 6,
                      multi_sentence: bool = True, seed: int = 0) -> list[HeadPrompt]:
    archetypes = archetypes or DEFAULT_ARCHETYPES
    prompts: list[HeadPrompt] = []
    for k, arch in enumerate(archetypes):
        examples = synthetic_head_dataset(arch, n_examples=n_examples, noise=0.0,
                                          multi_sentence=multi_sentence, seed=seed + k)
        expected = TEMPLATE_NAMES.index(arch) if arch in TEMPLATE_NAMES else None
        prompts.append(HeadPrompt(head_id=f"synthetic:{arch}", examples=examples,
                                  expected_template=expected))
    return prompts


In [ ]:
%%writefile scripts/__init__.py
"""Runnable scripts: attention extraction, prompt building, and the TRL trainer."""


In [ ]:
%%writefile scripts/build_prompt.py
"""
Builds the synthesis prompt shown to the policy LM, reproducing the format
described in the paper (section 2.2):

  "our pipeline first extracts attention patterns, filtering for the top
  2.5% of attention weights by magnitude to isolate the most salient
  token-pair interactions. These filtered patterns are formatted as
  token-pair weight summaries and embedded into structured prompts
  (approximately 4,000 tokens in length). The synthesis agent S is
  provided access to NumPy, spaCy and NLTK."

Two entry points:
  - build_round0_prompt(example): the initial synthesis prompt, matching
    the paper's one-shot generation step.
  - build_refinement_prompt(example, prior_code, feedback): appends the
    previous attempt and the env's structured error feedback (worst-fit
    token pairs) to the prompt, extending the paper's *fixed one-round*
    refinement into an arbitrary-round MDP. Feed the full accumulated
    transcript as the "prompt" to a stock single-turn GRPO trainer -- see
    scripts/grpo_train.py for why that's the practical way to get
    multi-round refinement out of off-the-shelf TRL without a custom
    multi-turn rollout loop.
"""
from __future__ import annotations

from env.data import AttentionExample

SYSTEM_PREAMBLE = """You are analyzing a single attention head from a transformer language model. \
You will be shown its attention pattern on an example sentence: which tokens (queries) attend to \
which other tokens (keys), and how strongly. Your job is to write a Python function that \
approximates this attention pattern given only the input tokens -- no access to model weights.

Function signature (required, exact name):

    def predict_attention(tokens: list[str]) -> np.ndarray:
        n = len(tokens)
        attention = np.zeros((n, n))
        # attention[i, j] = how much query token i attends to key token j
        ...
        return attention

Constraints:
  - Only `np` (numpy) and `re` are available -- no other imports.
  - Causal heads should respect j <= i (queries cannot attend to future tokens); \
bidirectional heads may attend to any j.
  - Output does not need to be row-normalized; the harness will normalize for you.
  - Start with a one-line docstring naming the linguistic or positional pattern \
you believe this head implements.

Respond with ONLY the Python function, no explanation before or after."""


def _top_k_percent_edges(attention, tokens, top_pct: float = 2.5, max_edges: int = 40):
    """Filter for the top `top_pct`% of attention weights by magnitude,
    matching the paper's stated extraction step, capped at `max_edges` to
    keep the prompt within a practical token budget.
    """
    n = attention.shape[0]
    flat = attention.flatten()
    k = max(1, int(len(flat) * top_pct / 100))
    k = min(k, max_edges)
    top_idx = flat.argsort()[::-1][:k]
    edges = []
    for idx in top_idx:
        i, j = divmod(int(idx), n)
        edges.append((i, j, float(attention[i, j])))
    return edges


def _format_edges(edges, tokens) -> str:
    lines = []
    for i, j, w in edges:
        qi = tokens[i] if i < len(tokens) else "?"
        kj = tokens[j] if j < len(tokens) else "?"
        lines.append(f"  '{qi}'[{i}] -> '{kj}'[{j}]  ({w:.3f})")
    return "\n".join(lines)


def build_round0_prompt(example: AttentionExample, top_pct: float = 2.5) -> str:
    edges = _top_k_percent_edges(example.attention, example.tokens, top_pct=top_pct)
    token_str = " ".join(f"'{t}'[{i}]" for i, t in enumerate(example.tokens))
    edge_str = _format_edges(edges, example.tokens)
    return (
        f"{SYSTEM_PREAMBLE}\n\n"
        f"Head: {example.head_id}\n\n"
        f"Input tokens:\n  {token_str}\n\n"
        f"Top {top_pct}% attention edges by weight (query -> key):\n{edge_str}\n"
    )


def build_refinement_prompt(example: AttentionExample, prior_code: str, feedback: dict,
                             top_pct: float = 2.5) -> str:
    base = build_round0_prompt(example, top_pct=top_pct)
    if feedback.get("status") == "execution_failed":
        fb_str = f"Your previous attempt failed to execute: {feedback['error']}"
    else:
        worst = feedback.get("worst_edges", [])
        worst_str = "\n".join(
            f"  '{e['query']}'[{e['query_idx']}] -> '{e['key']}'[{e['key_idx']}]: "
            f"real={e['real']:.3f}, your program predicted={e['predicted']:.3f}"
            for e in worst
        )
        fb_str = (
            f"Your previous attempt scored IoU={feedback.get('iou', 0):.3f}, "
            f"JSD={feedback.get('jsd', 0):.3f}.\n"
            f"Worst-predicted token pairs (largest error vs. real attention):\n{worst_str}"
        )
    return (
        f"{base}\n\n"
        f"--- Your previous attempt ---\n{prior_code}\n\n"
        f"--- Feedback ---\n{fb_str}\n\n"
        f"Write a revised version of predict_attention that fixes these errors. "
        f"Respond with ONLY the revised Python function."
    )


In [ ]:
%%writefile scripts/extract_attention_maps.py
"""
Extract per-head attention matrices from a real HF model and write them to
the JSONL schema `env/data.py::load_dataset` expects.

NOT RUNNABLE IN THIS SANDBOX -- torch/transformers aren't installed here
(no network access to fetch them or model weights). This is written against
the standard HF `output_attentions=True` API and should run as-is in a
normal GPU environment; if the original repo's own notebooks serialize
attention differently, treat that as the source of truth over this script's
assumptions and adjust `load_dataset` in env/data.py accordingly (that
function is deliberately the one seam to edit).

Usage:
    python scripts/extract_attention_maps.py \
        --model gpt2 --dataset tinystories --n-examples 200 \
        --out data/gpt2_attention.jsonl

Mirrors the paper's own choice of TinyStories as the extraction corpus
(section 2.3: "selected for its relative simplicity ... simplifies the act
of isolating specific head behaviors").
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from datasets import load_dataset as hf_load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


def extract_for_model(model_name: str, sentences: list[str], out_path: Path,
                       max_length: int = 64, device: str = "cpu") -> None:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, output_attentions=True, attn_implementation="eager"
    ).to(device).eval()

    n_layers = model.config.num_hidden_layers
    n_heads = model.config.num_attention_heads

    out_path.parent.mkdir(parents=True, exist_ok=True)
    n_written = 0
    with open(out_path, "w") as f, torch.no_grad():
        for sentence in sentences:
            enc = tokenizer(sentence, return_tensors="pt", truncation=True,
                             max_length=max_length).to(device)
            tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
            n = len(tokens)
            if n < 3:
                continue

            outputs = model(**enc)
            # outputs.attentions: tuple of (n_layers) tensors, each
            # (batch=1, n_heads, seq, seq)
            for layer_idx, layer_attn in enumerate(outputs.attentions):
                for head_idx in range(n_heads):
                    A = layer_attn[0, head_idx].cpu().numpy()  # (n, n)
                    head_id = f"{model_name}:L{layer_idx}H{head_idx}"
                    f.write(json.dumps({
                        "head_id": head_id,
                        "tokens": tokens,
                        "attention": A.tolist(),
                    }) + "\n")
                    n_written += 1
    print(f"wrote {n_written} (sentence x head) examples across "
          f"{n_layers} layers x {n_heads} heads to {out_path}")


def load_tinystories_sentences(n_examples: int, split: str = "train") -> list[str]:
    ds = hf_load_dataset("roneneldan/TinyStories", split=split, streaming=True)
    sentences = []
    for ex in ds:
        text = ex["text"].strip()
        if text:
            sentences.append(text)
        if len(sentences) >= n_examples:
            break
    return sentences


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="HF model id, e.g. gpt2, TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    ap.add_argument("--dataset", default="tinystories", choices=["tinystories"])
    ap.add_argument("--n-examples", type=int, default=200)
    ap.add_argument("--out", required=True)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    if args.dataset == "tinystories":
        sentences = load_tinystories_sentences(args.n_examples)
    else:
        raise ValueError(args.dataset)

    extract_for_model(args.model, sentences, Path(args.out), device=args.device)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/grpo_train.py
"""
GRPO training loop via TRL's library GRPOTrainer: policy LM writes candidate
attention-programs, reward comes from AttentionProgramEnv (IoU / JSD /
collapse-penalty), verifiable and non-LM-judged.

This is the LIBRARY-BACKED alternative to the from-scratch trainer in
train_grpo_torch.py (grpo/core.py). The from-scratch path is the one that
runs, is tested, and implements the GRPO objective by hand; this one hands
the optimizer step to TRL and is the sensible choice once you want to scale
to a large policy LM with TRL's vLLM/accelerate integration. Keep both: the
custom loop for control and understanding, TRL for scale.

This targets the paper's own stated gap directly (section "Limitations"):
they use one-shot generation plus a single fixed refinement round and say
closing the remaining fit gap "likely requires richer synthesis strategies
such as multi-round refinement with stronger feedback signals." Here, the
refinement policy itself is learned via group-relative RL instead of being
a fixed heuristic pass.

Verified against trl==1.2.0's GRPOConfig/GRPOTrainer signature (note:
`max_prompt_length` was removed from GRPOConfig in this line of releases, so
it's gone below). Re-check `pip show trl` before running a different version;
the reward_funcs kwarg-passing convention in particular drifts across
releases.

--- Multi-round refinement, practically ---
TRL's stock single-turn GRPOTrainer samples a group of G completions per
prompt, scores each, computes group-relative advantages, and updates. Two
ways to get multi-round refinement out of that:

  1. (what this script does) Treat "prompt" as the full accumulated
     transcript. Each training example already bakes in a fixed number of
     rounds: round 0 prompt -> round-0 completion -> env feedback -> round-1
     prompt (round 0 + feedback appended) -> ... -> final completion is what
     gets rewarded. You're training the policy to produce a GOOD FINAL
     answer given a fixed refinement history, which is learnable and
     compatible with off-the-shelf GRPOTrainer, but the history itself
     isn't produced by the policy being trained (it's built from a
     reference/frozen model's earlier attempts, or from the paper's own
     library as a warm-start). Simple, but the training signal never
     touches "how to refine your OWN mistake."
  2. (more faithful, more work) A genuine multi-turn RL loop: at each round,
     sample from the CURRENT policy, step the env, get feedback, append to
     context, and apply GRPO's advantage computation across full trajectories
     (multiple rounds) rather than single completions. trl==1.2.0's
     GRPOTrainer exposes `rollout_func` and `environment_factory` hooks for
     exactly this -- wrap AttentionProgramEnv (reset/step) in an
     environment_factory. `run_multiturn_grpo_step` below sketches the raw
     rollout shape (also directly usable with the from-scratch trainer);
     grpo/core.py's train_grpo already runs single-turn end to end.

This script implements option 1 end-to-end and points option 2 at TRL's
env hooks.
"""
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from env import AttentionProgramEnv, group_by_head, load_dataset
from env.attention_env import Action
from scripts.build_prompt import build_refinement_prompt, build_round0_prompt

CODE_BLOCK_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.DOTALL)


def extract_code(completion: str) -> str:
    """Pull a Python code block out of a raw LM completion. Falls back to
    the raw text if no fenced block is found (some models just emit code
    directly when told "respond with ONLY the function").
    """
    m = CODE_BLOCK_RE.search(completion)
    return m.group(1).strip() if m else completion.strip()


def build_reward_fn(envs_by_head: dict[str, AttentionProgramEnv]):
    """Returns a TRL-compatible reward function:
        reward_fn(prompts, completions, **kwargs) -> list[float]
    `kwargs` is expected to carry a parallel `head_id` list (TRL passes
    through any extra dataset columns as kwargs -- check your TRL version's
    exact convention; some pass them as `**kwargs`, older ones may need a
    custom collator). Each head gets its own env instance so val/fit splits
    and best-code tracking stay isolated per head.
    """
    def reward_fn(prompts, completions, head_id=None, **kwargs):
        assert head_id is not None, "dataset must carry a head_id column"
        rewards = []
        for completion, hid in zip(completions, head_id):
            code = extract_code(completion)
            env = envs_by_head[hid]
            env.reset()  # single-round scoring per completion here (option 1)
            result = env.step(Action(code=code))
            rewards.append(result.reward)
        return rewards
    return reward_fn


def build_dataset(data_path: str, max_heads: int | None = None):
    """Loads extracted attention maps, groups by head, builds one env per
    head, and returns (dataset_rows, envs_by_head). Each dataset row is a
    single round-0 synthesis prompt; extend with build_refinement_prompt
    calls chained onto a frozen reference model's completions if you want
    warm-started multi-round examples (see module docstring, option 1).
    """
    examples = load_dataset(data_path)
    grouped = group_by_head(examples)
    head_ids = list(grouped.keys())[:max_heads] if max_heads else list(grouped.keys())

    envs_by_head = {}
    rows = []
    for hid in head_ids:
        head_examples = grouped[hid]
        env = AttentionProgramEnv(head_examples, max_rounds=1, val_fraction=0.3)
        envs_by_head[hid] = env
        # use one representative fit example per head to build the prompt;
        # the env still scores against its own sampled example at reset()
        prompt = build_round0_prompt(head_examples[0])
        rows.append({"prompt": prompt, "head_id": hid})
    return rows, envs_by_head


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="JSONL from scripts/extract_attention_maps.py")
    ap.add_argument("--policy-model", default="Qwen/Qwen2.5-Coder-3B-Instruct")
    ap.add_argument("--max-heads", type=int, default=None)
    ap.add_argument("--num-generations", type=int, default=8, help="GRPO group size G")
    ap.add_argument("--output-dir", default="./grpo-attention-programs")
    ap.add_argument("--learning-rate", type=float, default=1e-6)
    args = ap.parse_args()

    # Deferred imports: only needed at actual training time, and aren't
    # installed in this build sandbox.
    from datasets import Dataset
    from trl import GRPOConfig, GRPOTrainer

    rows, envs_by_head = build_dataset(args.data, max_heads=args.max_heads)
    dataset = Dataset.from_list(rows)
    reward_fn = build_reward_fn(envs_by_head)

    config = GRPOConfig(
        output_dir=args.output_dir,
        learning_rate=args.learning_rate,
        num_generations=args.num_generations,
        # Keep completions short -- these are single functions, not essays.
        max_completion_length=512,
        # log_completions helps a lot here: eyeball whether the policy is
        # actually writing plausible attention logic vs. reward-hacking the
        # collapse penalty with superficially "varied" but meaningless code.
        log_completions=True,
    )

    trainer = GRPOTrainer(
        model=args.policy_model,
        reward_funcs=[reward_fn],
        args=config,
        train_dataset=dataset,
    )
    trainer.train()

    # After training: for each head, take the policy's best sampled program
    # (env tracks `_best_code` / `_best_reward` internally per env instance
    # across whatever steps were called against it) and evaluate on held-out.
    for hid, env in envs_by_head.items():
        if env._best_code is not None:
            stats = env.evaluate_held_out(env._best_code)
            print(f"{hid}: held-out mean_iou={stats['mean_iou']:.3f} "
                  f"fail_rate={stats['fail_rate']:.2f}")


def run_multiturn_grpo_step(policy_generate_fn, env: AttentionProgramEnv,
                             example, group_size: int = 8):
    """Sketch of option 2 (genuine multi-turn refinement) -- NOT a complete
    trainer, just the rollout shape you'd wrap a custom GRPO update around.

    policy_generate_fn: callable(prompt: str) -> str, e.g. a vLLM or HF
        generate() call against the policy being trained.
    Returns: list of (full_transcript, final_code, total_reward) trajectories,
        one per group member, ready for whatever advantage/loss computation
        your custom trainer implements (group-relative on `total_reward`).
    """
    trajectories = []
    for _ in range(group_size):
        obs = env.reset(example=example)
        prompt = build_round0_prompt(example)
        transcript = prompt
        total_reward = 0.0
        code = ""
        while True:
            completion = policy_generate_fn(transcript)
            code = extract_code(completion)
            result = env.step(Action(code=code))
            total_reward += result.reward
            if result.done:
                break
            transcript = build_refinement_prompt(example, code, result.observation.feedback)
        trajectories.append((transcript, code, total_reward))
    return trajectories


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/__init__.py
"""Test suite: env/reward (numpy-only) and the PyTorch GRPO trainer."""


In [ ]:
%%writefile tests/test_env.py
"""
Sanity tests against synthetic ground-truth heads. No torch, no model
weights, no network -- these should pass anywhere numpy is installed, and
are meant to be the first thing you run after cloning, before spending any
API budget on a real policy LM.

Run: python -m pytest tests/test_env.py -v
  or: python tests/test_env.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from env import (
    Action, AttentionProgramEnv, compute_reward, iou_score, jsd_score,
    synthetic_head_dataset,
)
from env.executor import run_program


PREVIOUS_TOKEN_PROGRAM = """
import numpy as np

def predict_attention(tokens):
    n = len(tokens)
    attention = np.zeros((n, n))
    for i in range(n):
        j = max(i - 1, 0)
        attention[i, j] = 0.92
        attention[i, i] += 0.08
    row_sums = attention.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return attention / row_sums
"""

FIRST_TOKEN_PROGRAM = """
import numpy as np

def predict_attention(tokens):
    n = len(tokens)
    attention = np.zeros((n, n))
    attention[:, 0] = 1.0
    return attention
"""

RANDOM_PROGRAM = """
import numpy as np

def predict_attention(tokens):
    n = len(tokens)
    rng = np.random.default_rng(42)
    m = rng.random((n, n))
    return m / m.sum(axis=1, keepdims=True)
"""

SYNTAX_ERROR_PROGRAM = "def predict_attention(tokens:\n    return None"

UNSAFE_PROGRAM = """
import os

def predict_attention(tokens):
    os.system("echo pwned")
    return None
"""

SENTENCE_BOUNDARY_PROGRAM = """
import numpy as np

def predict_attention(tokens):
    n = len(tokens)
    attention = np.zeros((n, n)) + 0.01
    for i in range(n):
        for j in range(i + 1):
            is_start = (j == 0) or (tokens[j - 1] in {'.', '!', '?'})
            if is_start and tokens[j] not in {'.', '!', '?', ','}:
                attention[i, j] += 3.0
        attention[i, i] += 0.5
    row_sums = attention.sum(axis=1, keepdims=True)
    return attention / row_sums
"""


def test_correct_previous_token_program_scores_high():
    data = synthetic_head_dataset("previous_token", n_examples=10, noise=0.0)
    env = AttentionProgramEnv(data, max_rounds=1, val_fraction=0.3)
    obs = env.reset(example=data[0])
    result = env.step(Action(code=PREVIOUS_TOKEN_PROGRAM))
    assert result.info["executable"], result.info["error"]
    assert result.info["iou"] > 0.75, f"expected high IoU, got {result.info['iou']}"
    print(f"[previous_token, correct program] reward={result.reward:.3f} "
          f"iou={result.info['iou']:.3f} jsd={result.info['jsd']:.3f} "
          f"complexity_penalty={result.info['complexity_penalty']:.3f}")


def test_wrong_archetype_scores_low():
    data = synthetic_head_dataset("previous_token", n_examples=10, noise=0.0)
    env = AttentionProgramEnv(data, max_rounds=1, val_fraction=0.3)
    env.reset(example=data[0])
    result = env.step(Action(code=FIRST_TOKEN_PROGRAM))
    print(f"[previous_token, first-token program] reward={result.reward:.3f} "
          f"iou={result.info['iou']:.3f}")
    assert result.info["iou"] < 0.5


def test_random_program_scores_lowest():
    data = synthetic_head_dataset("previous_token", n_examples=10, noise=0.0)
    env = AttentionProgramEnv(data, max_rounds=1, val_fraction=0.3)
    env.reset(example=data[0])
    correct = env.step(Action(code=PREVIOUS_TOKEN_PROGRAM))

    env.reset(example=data[0])
    rand = env.step(Action(code=RANDOM_PROGRAM))
    print(f"[previous_token] correct reward={correct.reward:.3f} vs "
          f"random reward={rand.reward:.3f}")
    assert correct.reward > rand.reward


def test_syntax_error_gets_floor_reward():
    data = synthetic_head_dataset("previous_token", n_examples=5, noise=0.0)
    env = AttentionProgramEnv(data, max_rounds=1, val_fraction=0.3)
    env.reset(example=data[0])
    result = env.step(Action(code=SYNTAX_ERROR_PROGRAM))
    assert not result.info["executable"]
    assert result.reward == -1.0
    print(f"[syntax error] reward={result.reward} (expected floor -1.0)")


def test_unsafe_program_is_rejected_not_executed():
    data = synthetic_head_dataset("previous_token", n_examples=5, noise=0.0)
    env = AttentionProgramEnv(data, max_rounds=1, val_fraction=0.3)
    env.reset(example=data[0])
    result = env.step(Action(code=UNSAFE_PROGRAM))
    assert not result.info["executable"]
    assert "unsafe" in (result.info["error"] or "").lower()
    print(f"[unsafe program] correctly rejected: {result.info['error']}")


def _multi_sentence_example():
    # Two sentences in one example -- deliberately breaks the degenerate
    # equivalence between "sentence boundary" and "first token" that holds
    # on any single-sentence input (both start-tokens coincide at index 0).
    from env.data import AttentionExample, _sentence_boundary_attention
    tokens = "Tim was happy . He ran outside and played with his dog .".split(" ")
    attn = _sentence_boundary_attention(tokens)
    return AttentionExample(head_id="synthetic:sentence_boundary_multi",
                             tokens=tokens, attention=attn)


def test_trivial_first_token_penalized_even_with_comparable_iou():
    """This is the paper's own flagged failure mode: a policy optimizing IoU
    alone can collapse onto degenerate first-token attenders. Uses a
    multi-sentence example where "attend to token 0" and "attend to sentence
    starts" genuinely diverge (unlike single-sentence inputs, where they're
    accidentally identical), then checks the content-invariance probe
    actually suppresses the trivial program relative to a structured one.
    """
    ex = _multi_sentence_example()
    env = AttentionProgramEnv([ex], max_rounds=1, val_fraction=0.99)

    env.reset(example=ex)
    trivial = env.step(Action(code=FIRST_TOKEN_PROGRAM))

    env.reset(example=ex)
    structured = env.step(Action(code=SENTENCE_BOUNDARY_PROGRAM))

    print(f"[multi-sentence] trivial first-token: iou={trivial.info['iou']:.3f} "
          f"reward={trivial.reward:.3f} | structured: iou={structured.info['iou']:.3f} "
          f"reward={structured.reward:.3f}")
    assert structured.info["iou"] > trivial.info["iou"], (
        "structured program should fit the two-sentence-start pattern "
        "better than a program that only ever looks at index 0"
    )
    assert structured.reward > trivial.reward


def test_multi_round_refinement_feedback_and_best_tracking():
    data = synthetic_head_dataset("previous_token", n_examples=10, noise=0.0)
    env = AttentionProgramEnv(data, max_rounds=3, val_fraction=0.3)
    obs = env.reset(example=data[0])
    assert obs.feedback is None and obs.round_idx == 0

    r1 = env.step(Action(code=RANDOM_PROGRAM))
    assert not r1.done
    assert r1.observation.feedback is not None
    assert r1.observation.feedback["status"] == "scored"
    assert "worst_edges" in r1.observation.feedback
    print(f"[round 1] feedback worst edge example: {r1.observation.feedback['worst_edges'][0]}")

    r2 = env.step(Action(code=PREVIOUS_TOKEN_PROGRAM))
    assert not r2.done
    assert r2.info["best_reward_so_far"] >= r1.reward

    r3 = env.step(Action(code=PREVIOUS_TOKEN_PROGRAM))
    assert r3.done
    assert r3.observation is None
    print(f"[round 3, terminal] best_reward_so_far={r3.info['best_reward_so_far']:.3f}")


def test_held_out_evaluation_generalizes():
    data = synthetic_head_dataset("previous_token", n_examples=20, noise=0.01)
    env = AttentionProgramEnv(data, max_rounds=1, val_fraction=0.4)
    stats = env.evaluate_held_out(PREVIOUS_TOKEN_PROGRAM)
    print(f"[held-out eval] mean_iou={stats['mean_iou']:.3f} "
          f"mean_jsd={stats['mean_jsd']:.3f} fail_rate={stats['fail_rate']:.2f}")
    assert stats["mean_iou"] > 0.7
    assert stats["fail_rate"] == 0.0


def test_iou_and_jsd_metrics_directly():
    import numpy as np
    A = np.array([[1.0, 0.0], [0.0, 1.0]])
    B = np.array([[1.0, 0.0], [0.0, 1.0]])
    assert abs(iou_score(A, B) - 1.0) < 1e-9
    assert jsd_score(A, B) < 1e-9

    C = np.array([[0.0, 1.0], [1.0, 0.0]])
    assert iou_score(A, C) < 1e-9
    assert jsd_score(A, C) > 0.6  # near ln(2), fully disjoint distributions


if __name__ == "__main__":
    tests = [v for k, v in list(globals().items()) if k.startswith("test_")]
    passed, failed = 0, 0
    for t in tests:
        try:
            t()
            passed += 1
            print(f"PASS: {t.__name__}\n")
        except AssertionError as e:
            failed += 1
            print(f"FAIL: {t.__name__}: {e}\n")
    print(f"\n{passed} passed, {failed} failed out of {len(tests)}")
    if failed:
        sys.exit(1)


In [ ]:
%%writefile tests/test_grpo_torch.py
"""
Tests for the from-scratch PyTorch GRPO trainer (grpo/).

Covers three layers, all offline (no model download, no network):
  - the program DSL compiles to executable programs and separates archetypes;
  - the GRPO primitives (group advantages, clipped loss) are correct;
  - the *toy* backend actually trains -- reward climbs and the policy learns
    the right program per head archetype;
  - the *HF causal-LM* backend runs end to end on a locally-built tiny GPT-2:
    the loop steps, log-probs are differentiable and wired to the optimizer,
    and old/new log-probs agree at inner-epoch 0 (importance ratio == 1).

Run: python -m pytest tests/test_grpo_torch.py -v
  or: python tests/test_grpo_torch.py
"""
import os
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import pytest

torch = pytest.importorskip("torch")

from grpo.core import GRPOConfig, compute_group_advantages, grpo_loss, train_grpo
from grpo.program_dsl import (
    NUM_SHARPNESS, NUM_TEMPLATES, compile_action, mean_features,
)
from grpo.rewarding import ProgramScorer
from grpo.toy_data import build_toy_prompts


# --------------------------------------------------------------------------
# Program DSL
# --------------------------------------------------------------------------

def test_every_dsl_action_compiles_and_executes():
    from env import run_program_inproc
    tokens = "Tim was happy . He ran outside .".split(" ")
    for t in range(NUM_TEMPLATES):
        for s in range(NUM_SHARPNESS):
            A, ok, err = run_program_inproc(compile_action(t, s), tokens)
            assert ok, f"action ({t},{s}) failed: {err}"
            assert A.shape == (len(tokens), len(tokens))


def test_matching_template_wins_per_archetype():
    """The verifiable reward should rank a head's own template highest, so
    there is a real signal for the policy to learn."""
    scorer = ProgramScorer(trusted=True)
    for p in build_toy_prompts():
        rewards = {(t, s): scorer.mean_reward(compile_action(t, s), p.examples)
                   for t in range(NUM_TEMPLATES) for s in range(NUM_SHARPNESS)}
        best_t = max(rewards, key=rewards.get)[0]
        assert best_t == p.expected_template, (
            f"{p.head_id}: best template {best_t} != expected {p.expected_template}")


# --------------------------------------------------------------------------
# GRPO primitives
# --------------------------------------------------------------------------

def test_group_advantages_are_standardized_per_group():
    rewards = torch.tensor([0.0, 1.0, 2.0, 10.0, 10.0, 10.0])
    groups = [0, 0, 0, 1, 1, 1]
    adv = compute_group_advantages(rewards, groups, eps=1e-6)
    # group 0 standardized to ~zero mean, unit std; group 1 all-equal -> ~0
    assert abs(float(adv[:3].mean())) < 1e-5
    assert abs(float(adv[:3].std(unbiased=False)) - 1.0) < 1e-3
    assert torch.allclose(adv[3:], torch.zeros(3), atol=1e-4)


def test_grpo_loss_finite_and_entropy_reduces_loss():
    logps = [torch.tensor([-1.0, -2.0], requires_grad=True)]
    old = [torch.tensor([-1.0, -2.0])]
    adv = torch.tensor([1.5])
    kl = [torch.tensor([0.1, 0.1])]
    ent = [torch.tensor([0.5, 0.5])]
    loss_no_ent, _ = grpo_loss(logps, old, adv, kl, 0.2, 0.02, ent, ent_coef=0.0)
    loss_ent, stats = grpo_loss(logps, old, adv, kl, 0.2, 0.02, ent, ent_coef=0.1)
    assert torch.isfinite(loss_ent)
    assert float(loss_ent.detach()) < float(loss_no_ent.detach())  # entropy bonus lowers loss
    assert stats["entropy"] > 0


# --------------------------------------------------------------------------
# Toy backend: end-to-end learning
# --------------------------------------------------------------------------

def test_toy_grpo_training_converges():
    from grpo import run_toy_training
    cfg = GRPOConfig(steps=60, group_size=12, lr=3e-3, kl_coef=0.02, ent_coef=0.02, seed=0)
    out = run_toy_training(cfg=cfg, verbose=False)
    hist = out["history"]
    init, final = hist[0]["mean_reward"], hist[-1]["mean_reward"]
    assert final > init + 0.4, f"reward barely moved: {init:.3f} -> {final:.3f}"
    assert final > 0.5, f"final reward too low: {final:.3f}"
    assert out["accuracy"] >= 0.75, f"template accuracy too low: {out['accuracy']}"
    # every rollout executed (DSL programs are always well-formed)
    assert all(h["frac_executable"] == 1.0 for h in hist)


# --------------------------------------------------------------------------
# HF backend: runs offline on a tiny locally-built GPT-2
# --------------------------------------------------------------------------

def _tiny_hf_policy(prompts, max_new_tokens=16):
    transformers = pytest.importorskip("transformers")
    from grpo.hf_policy import CharTokenizer, HFPolicy
    torch.manual_seed(0)
    tok = CharTokenizer()
    cfg = transformers.GPT2Config(vocab_size=tok.vocab_size, n_positions=256,
                                  n_embd=32, n_layer=2, n_head=2)
    model = transformers.GPT2LMHeadModel(cfg)
    model.config.pad_token_id = tok.pad_token_id
    return HFPolicy(model, tok, prompts, device="cpu", max_new_tokens=max_new_tokens)


def test_hf_backend_train_loop_runs_offline():
    from grpo import HeadPrompt, make_reward_fn
    from env.data import synthetic_head_dataset
    policy = _tiny_hf_policy(["Write predict_attention for head A:"])
    head_prompts = [HeadPrompt("A", synthetic_head_dataset("previous_token", n_examples=3, noise=0.0))]
    reward_fn = make_reward_fn(head_prompts, ProgramScorer(trusted=False))
    hist = train_grpo(policy, 1, reward_fn, GRPOConfig(steps=2, group_size=4, seed=0))
    assert len(hist) == 2
    assert all(torch.isfinite(torch.tensor(h["loss"])) for h in hist)


def test_hf_backend_is_differentiable_and_ratio_one_at_epoch0():
    policy = _tiny_hf_policy(["Write predict_attention:"])
    rollouts = policy.sample([0], 4)
    new_logps, kls, ent = policy.evaluate(rollouts)

    # (a) old == new log-probs at epoch 0 -> importance ratio is exactly 1
    for nl, r in zip(new_logps, rollouts):
        assert torch.allclose(nl, r.old_logps, atol=1e-5)

    # (b) with an injected non-zero advantage, a step actually moves params
    adv = torch.tensor([1.0, -1.0, 1.0, -1.0])
    loss, _ = grpo_loss(new_logps, [r.old_logps for r in rollouts], adv, kls, 0.2, 0.02, ent, 0.0)
    assert torch.isfinite(loss)
    opt = torch.optim.Adam(policy.parameters(), lr=1e-3)
    before = [p.detach().clone() for p in policy.parameters()]
    opt.zero_grad()
    loss.backward()
    opt.step()
    moved = sum(float((a - b).abs().sum()) for a, b in zip(policy.parameters(), before))
    assert moved > 0, "optimizer step did not change any parameters"


if __name__ == "__main__":
    fns = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    passed, failed = 0, 0
    for t in fns:
        try:
            t()
            passed += 1
            print(f"PASS: {t.__name__}")
        except Exception as e:  # noqa: BLE001
            failed += 1
            print(f"FAIL: {t.__name__}: {type(e).__name__}: {e}")
    print(f"\n{passed} passed, {failed} failed out of {len(fns)}")
    if failed:
        sys.exit(1)


In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
os.environ["HF_HUB_OFFLINE"] = "0"      # we download GPT-2 and Qwen3-4B
os.environ["TRANSFORMERS_OFFLINE"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import env, grpo
print("imported env + grpo OK")

## 2. Extract real attention heads from GPT-2

We run GPT-2 on TinyStories sentences (the paper's own extraction corpus) with
`output_attentions=True`, giving a real `n×n` attention matrix per
`(sentence, layer, head)`. We then keep the **most structured** heads — the
ones whose attention is peaked enough to plausibly implement a rule worth
explaining (diffuse heads have no pattern to synthesize).

In [ ]:
from pathlib import Path
import numpy as np
from scripts.extract_attention_maps import extract_for_model, load_tinystories_sentences
from env import load_dataset, group_by_head

N_SENTENCES = 30          # extraction corpus size (knob)
N_HEADS      = 12         # how many heads to run the experiment on (knob)

sents = load_tinystories_sentences(N_SENTENCES)
extract_for_model("gpt2", sents, Path("data/gpt2_attn.jsonl"), device=DEVICE)
grouped = group_by_head(load_dataset("data/gpt2_attn.jsonl"))
print("extracted", len(grouped), "heads x", N_SENTENCES, "sentences")

In [ ]:
# Rank heads by "peakedness" = average over examples of the mean top-attention
# per query row. High = concentrated (a rule to explain); low = diffuse.
def peakedness(examples):
    return float(np.mean([e.attention.max(axis=1).mean() for e in examples]))

ranked = sorted(grouped.items(), key=lambda kv: -peakedness(kv[1]))
selected = ranked[:N_HEADS]
print("selected heads (most structured):")
for hid, exs in selected:
    print(f"  {hid:14s}  peakedness={peakedness(exs):.3f}  ({len(exs)} examples)")

## 3. Load the synthesizer LM (Qwen3-4B)

Inference-only, so we just need the model in bf16 — no LoRA, no optimizer.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

SYNTH_MODEL = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(SYNTH_MODEL)
try:
    model = AutoModelForCausalLM.from_pretrained(SYNTH_MODEL, dtype=torch.bfloat16, device_map="auto")
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(SYNTH_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model.eval(); model.config.use_cache = True
print(SYNTH_MODEL, "loaded |", round(sum(p.numel() for p in model.parameters())/1e9, 2), "B params")

## 4. The experiment: one-shot vs. N-round refinement

For each head:
- **round 0** is the one-shot baseline: sample `N` programs from the round-0
  prompt, keep the best by verifiable reward.
- each subsequent round rebuilds the prompt with the env's **worst-edge
  feedback** on the running-best program (`build_refinement_prompt`) and samples
  `N` more, keeping the best-so-far.

We record, for the one-shot best and the multi-round best: mean IoU / JSD, the
degeneracy score, held-out IoU (generalization to unseen sentences), and
whether it executes.

In [ ]:
from grpo.hf_policy import extract_code
from scripts.build_prompt import build_round0_prompt, build_refinement_prompt
from grpo.rewarding import ProgramScorer
from env import run_program_inproc, compute_reward
from env.reward import positional_collapse_score
from env.attention_env import AttentionProgramEnv

N_SAMPLES = 6      # best-of-N per round (knob)
ROUNDS    = 3      # refinement rounds incl. round 0 (knob)

scorer = ProgramScorer(trusted=True)  # in-process exec; isolated Modal container

def to_chat(user_msg):
    msgs = [{"role": "user", "content": user_msg}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_programs(user_msg, n=N_SAMPLES, max_new_tokens=768, temperature=0.6):
    enc = tokenizer(to_chat(user_msg), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, do_sample=True, num_return_sequences=n,
                         max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.95,
                         pad_token_id=tokenizer.eos_token_id)
    plen = enc["input_ids"].shape[1]
    return [extract_code(tokenizer.decode(seq[plen:], skip_special_tokens=True)) for seq in out]

def metrics(code_str, examples, env):
    ex = examples[0]
    A, ok, err = run_program_inproc(code_str, ex.tokens)
    if not ok:
        return {"iou": 0.0, "jsd": float(np.log(2)), "degeneracy": 1.0,
                "heldout_iou": 0.0, "executable": False}
    ious = []
    for e in examples:
        AA, ook, _ = run_program_inproc(code_str, e.tokens)
        if ook:
            ious.append(compute_reward(e.attention, AA, code_str, ook).iou)
    ho = env.evaluate_held_out(code_str)
    return {"iou": float(np.mean(ious)) if ious else 0.0,
            "jsd": float(compute_reward(ex.attention, A, code_str, ok).jsd),
            "degeneracy": positional_collapse_score(A),
            "heldout_iou": ho["mean_iou"], "executable": True}

def run_head(examples):
    env = AttentionProgramEnv(examples, max_rounds=ROUNDS, val_fraction=0.3, seed=0)
    ex = examples[0]
    prompt = build_round0_prompt(ex)
    best_code, best_reward, oneshot_code = None, -1e9, None
    per_round = []
    for rnd in range(ROUNDS):
        scored = [(scorer.mean_reward(c, examples), c) for c in generate_programs(prompt)]
        r, c = max(scored, key=lambda x: x[0])
        if rnd == 0:
            oneshot_code = c                       # round 0 == one-shot baseline
        if r > best_reward:
            best_reward, best_code = r, c
        per_round.append(best_reward)
        A, ok, err = run_program_inproc(best_code, ex.tokens)
        bd = compute_reward(ex.attention, A, best_code, ok, err)
        fb = AttentionProgramEnv._build_feedback(ex, A, bd)
        prompt = build_refinement_prompt(ex, best_code, fb)
    return metrics(oneshot_code, examples, env), metrics(best_code, examples, env), per_round, best_code

In [ ]:
rows, per_round_all, best_programs = [], [], {}
for hid, examples in selected:
    base_m, ref_m, per_round, code_str = run_head(examples)
    best_programs[hid] = code_str
    per_round_all.append(per_round)
    rows.append({"head": hid,
                 "oneshot_iou": base_m["iou"], "refined_iou": ref_m["iou"],
                 "oneshot_heldout": base_m["heldout_iou"], "refined_heldout": ref_m["heldout_iou"],
                 "oneshot_jsd": base_m["jsd"], "refined_jsd": ref_m["jsd"],
                 "oneshot_deg": base_m["degeneracy"], "refined_deg": ref_m["degeneracy"]})
    print(f"{hid:14s} IoU one-shot {base_m['iou']:.3f} -> refined {ref_m['iou']:.3f}"
          f"  | held-out {base_m['heldout_iou']:.3f} -> {ref_m['heldout_iou']:.3f}")

## 5. Results

The summary table is the headline: one-shot vs. multi-round refinement,
averaged over the real heads, on fit **and held-out** IoU, plus the
**degeneracy rate** — the "pruning-effect" confound the paper flags but never
measures.

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)

DEG_TAU = 0.5
summary = pd.DataFrame({
    "metric": ["mean IoU (fit)", "mean IoU (held-out)", "mean JSD",
               f"degeneracy rate (>{DEG_TAU})"],
    "one-shot": [df.oneshot_iou.mean(), df.oneshot_heldout.mean(),
                 df.oneshot_jsd.mean(), (df.oneshot_deg > DEG_TAU).mean()],
    "N-round refine": [df.refined_iou.mean(), df.refined_heldout.mean(),
                       df.refined_jsd.mean(), (df.refined_deg > DEG_TAU).mean()],
})
summary["delta"] = summary["N-round refine"] - summary["one-shot"]
print(summary.to_string(index=False))
df.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

# (a) per-head one-shot vs refined held-out IoU
x = np.arange(len(df)); w = 0.4
ax[0].bar(x - w/2, df.oneshot_heldout, w, label="one-shot")
ax[0].bar(x + w/2, df.refined_heldout, w, label=f"{ROUNDS}-round refine")
ax[0].set_xticks(x); ax[0].set_xticklabels(df.head, rotation=90, fontsize=7)
ax[0].set_ylabel("held-out IoU"); ax[0].set_title("Per-head fit: one-shot vs refinement")
ax[0].legend()

# (b) refinement improvement curve (mean best reward vs round)
maxlen = max(len(p) for p in per_round_all)
padded = np.array([p + [p[-1]] * (maxlen - len(p)) for p in per_round_all])
ax[1].plot(range(maxlen), padded.mean(0), marker="o")
ax[1].fill_between(range(maxlen), padded.mean(0) - padded.std(0),
                   padded.mean(0) + padded.std(0), alpha=0.2)
ax[1].set_xlabel("refinement round"); ax[1].set_ylabel("mean best reward")
ax[1].set_title("Refinement improves the best program per round")
plt.tight_layout(); plt.show()

## 6. Reading the result

- If **refinement > one-shot** on held-out IoU, that is direct evidence for the
  paper's hypothesis that "richer synthesis strategies such as multi-round
  refinement with stronger feedback signals" close the fit gap — now *measured*
  on real heads.
- The **degeneracy rate** row is the novel contribution: how often each method's
  best-by-IoU program is actually a content-blind shortcut
  (`positional_collapse_score` > τ). Lower is better.
- Honest caveats: single synthesizer model, GPT-2 as the target model, a
  modest head count, IoU/JSD fit rather than causal swap-in (Next-steps #3).
  Scale `N_HEADS`, `N_SAMPLES`, `ROUNDS` up for a publishable version, and add
  multiple seeds for error bars.